<img src="https://raw.githubusercontent.com/jwohlwend/boltz/main/docs/boltz2_title.png" height="200" align="right" style="height:240px">

## Boltz-2: Democratizing Biomolecular Interaction Modeling

## CSV Batch Processor for High-Throughput Predictions

Easy-to-use batch processing interface for [Boltz-2](https://doi.org/10.1101/2025.06.14.659707), a biomolecular foundation model that jointly models complex structures and binding affinities. Boltz-2 approaches [AlphaFold3](https://www.nature.com/articles/s41586-024-07487-w) accuracy while running 1000x faster than physics-based methods.

### **Key Features:**
- **Structure Prediction**: Protein, DNA, RNA, and ligand complexes with AlphaFold3-level accuracy
- **Binding Affinity**: First deep learning model to approach FEP accuracy for drug discovery
- **Batch Processing**: CSV-based workflow for processing hundreds of predictions efficiently
- **Post-Translational Modifications**: Built-in support for 15 common PTMs plus custom modifications
- **Advanced Constraints**: Pocket binding constraints and covalent bond specifications
- **Automatic MSA Generation**: Integrated ColabFold MSA server for protein sequences
- **Google Drive Integration**: Automatic upload of results to your Drive
- **Open Source**: MIT license for academic and commercial use

### **Workflow:**
1. **Prepare CSV**: Create input file with sequences, modifications, and constraints
2. **Configure**: Set prediction parameters (recycling, sampling, affinity, precision)
3. **Run Batch**: Automated processing with progress tracking
4. **Download**: Results automatically uploaded to Google Drive

### **CSV Input Format:**
- **Required columns**: `jobname`, `seq1_name`, `seq1_type`, `seq1`
- **Optional columns**: `seq1_copies`, `seq1_mods`, `pocket_binder`, `pocket_contacts`, `covalent_bonds`
- **Supports**: Up to 10 sequences per job (seq1 through seq10)
- **Sequence types**: protein, dna, rna, ccd (ligand), smiles (ligand)

### **Supported Modifications:**
- **PTMs**: Phosphorylation (SEP, TPO, PTR), Methylation (MLY, M3L), Acetylation (ALY), and more
- **Ligands**: ATP, GTP, NAD, FAD, SAM, Heme, and 20+ common molecules
- **Ions**: Ca2+, Mg2+, Zn2+, Fe2+/3+, and other metal ions
- **Glycans**: NAG, MAN, GAL, FUC, and other carbohydrate modifications
- **DNA/RNA Mods**: Methylation, oxidation, and other nucleotide modifications
- **Custom**: Upload your own reference CSV with additional modifications

### **Repository:**
- [Boltz-2 GitHub Repository](https://github.com/jwohlwend/boltz)
- [Boltz-2 Colab Repository](https://github.com/JKourelis/Colab_Boltz-2)

### **Citations:**

**Boltz-2:**

[Passaro S, Corso G, Wohlwend J, et al. Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction. *bioRxiv*, 2025](https://doi.org/10.1101/2025.06.14.659707)

**Boltz-1:**

[Wohlwend J, Corso G, Passaro S, et al. Boltz-1: Democratizing Biomolecular Interaction Modeling. *bioRxiv*, 2024](https://doi.org/10.1101/2024.11.19.624167)

**If using automatic MSA generation:**

[Mirdita M, Sch\u00fctze K, Moriwaki Y, et al. ColabFold: making protein folding accessible to all. *Nature Methods*, 2022](https://doi.org/10.1038/s41592-022-01488-1)

---

### **Quick Start Guide:**

1. **Run Cell 1**: Kernel restart (if needed \u2014 skips if already installed)
2. **Run Cell 2**: Install Boltz-2 (skips if already installed)
3. **Run Cell 3**: Initialize CSV processor
4. **Run Cell 4**: Upload your CSV file and connect Google Drive
5. **Run Cell 5**: Configure prediction parameters
6. **Run Cell 6**: Execute batch predictions (results auto-uploaded to Drive)

**Example CSV structure:**
```csv
jobname,seq1_name,seq1_type,seq1,seq2_name,seq2_type,seq2
my_protein,proteinA,protein,MSEQVENCE...,ligandATP,ccd,ATP
complex_with_ptm,proteinB,protein,MKLLVV...,,,
```

For proteins with PTMs: `seq1_mods` column with format `POSITION:CCD` (e.g., `10:SEP,25:PTR`)

---

**Ready to start? Run Cell 1 below!**

In [ ]:
#@title Cell 1: Kernel Restart (if needed)
import subprocess
import sys
import os
import re

# Restart marker to handle Colab Feb 2025 NumPy issue
restart_marker = "/content/.boltz_numpy_restart"
is_post_restart = os.path.exists(restart_marker)

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"[{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {result.stderr[:300]}")
        return False
    print("OK")
    return True

def get_cuda_version():
    """Detect CUDA version from nvidia-smi"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                version = match.group(1)
                major = int(version.split('.')[0])
                minor = int(version.split('.')[1])
                return major, minor, version
    except Exception as e:
        print(f"{chr(0x26a0)}  Could not detect CUDA version: {e}")
    return None, None, None

def test_cuequivariance_kernels():
    """Test if cuEquivariance triangle kernels are available"""
    print("\n" + "=" * 60)
    print("CUEQUIVARIANCE KERNEL PREFLIGHT TEST")
    print("=" * 60)

    try:
        import torch
        print(f"{chr(0x2705)} PyTorch: {torch.__version__}")
        print(f"{chr(0x2705)} CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"{chr(0x2705)} CUDA version: {torch.version.cuda}")
            print(f"{chr(0x2705)} GPU: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"{chr(0x274c)} PyTorch check failed: {e}")
        return False

    # Test cuequivariance-torch import
    try:
        import cuequivariance_torch
        print(f"{chr(0x2705)} cuequivariance-torch installed")
    except ImportError as e:
        print(f"{chr(0x26a0)}  cuequivariance-torch not found: {e}")
        return False

    # Test cuequivariance-ops-torch-cu12 import
    try:
        import cuequivariance_ops_torch
        print(f"{chr(0x2705)} cuequivariance-ops-torch-cu12 installed")
    except ImportError as e:
        print(f"{chr(0x26a0)}  cuequivariance-ops-torch-cu12 not found: {e}")
        return False

    # CRITICAL TEST: triangle_multiplicative_update
    try:
        from cuequivariance_ops_torch.triangle import triangle_multiplicative_update
        print(f"{chr(0x2705)} triangle_multiplicative_update import: SUCCESS")

        if callable(triangle_multiplicative_update):
            print(f"{chr(0x2705)} triangle_multiplicative_update is callable")
        else:
            print(f"{chr(0x274c)} triangle_multiplicative_update exists but is not callable")
            return False

    except ImportError as e:
        print(f"{chr(0x274c)} triangle_multiplicative_update import FAILED: {e}")
        print(f"   This error requires --no_kernels flag")
        return False
    except Exception as e:
        print(f"{chr(0x274c)} Unexpected error testing triangle kernels: {e}")
        return False

    # Test triangle_attention_update
    try:
        from cuequivariance_ops_torch.triangle import triangle_attention_update
        print(f"{chr(0x2705)} triangle_attention_update import: SUCCESS")
    except ImportError as e:
        print(f"{chr(0x26a0)}  triangle_attention_update import failed: {e}")
        return False

    print("=" * 60)
    return True

if not is_post_restart:
    # PRE-RESTART: PREFLIGHT CHECKS
    print("=" * 60)
    print("PREFLIGHT CHECKS")
    print("=" * 60)

    # Check GPU
    cuda_major, cuda_minor, cuda_version = get_cuda_version()
    if cuda_major is None:
        print(f"{chr(0x274c)} No CUDA detected - cannot proceed")
        sys.exit(1)

    print(f"{chr(0x2705)} CUDA Version: {cuda_version}")
    print(f"   Driver CUDA: {cuda_major}.{cuda_minor}")

    # Check GPU type
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"{chr(0x2705)} GPU: {gpu_name}")

    print(f"\n{chr(0x1f4e6)} Will install: PyTorch 2.9.0 (pip resolves CUDA wheel automatically)")

    # Check pre-loaded NumPy
    result = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True)
    if result.returncode == 0:
        numpy_ver = result.stdout.strip()
        print(f"\n{chr(0x26a0)}  Pre-loaded NumPy: {numpy_ver}")
        if numpy_ver.startswith('2.'):
            print("   -> Colab Feb 2025 issue detected")
            print("   -> NumPy 2.x must be cleared from memory")

    print("\n" + "=" * 60)
    print("RESTARTING RUNTIME TO CLEAR NUMPY 2.X")
    print("=" * 60)
    print("Runtime will restart in 2 seconds")
    print("After restart: Run this cell again to install")
    print("=" * 60)

    with open(restart_marker, "w") as f:
        f.write("restarted")

    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)



In [ ]:
#@title Cell 2: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess
import sys
import os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
setup_google_drive = True #@param {type:"boolean"}
#@markdown - Setup Google Drive for automatic result upload

gdrive_folder_name = "Boltz2_Predictions" #@param {type:"string"}
#@markdown - Google Drive folder name for batch results

msa_folder = ""  #@param {type:"string"}
#@markdown > **Pre-computed MSAs** (from MSA Colab). Leave empty for online MSA.
#@markdown > Enter a path, zip filename, or Google Drive zip name. The notebook will
#@markdown > resolve it to the `paired/` directory automatically.

print("=" * 60)
print(f"{chr(0x1f4c1)} UPLOAD CSV AND CONNECT GOOGLE DRIVE")
print("=" * 60)

print(f"\n{chr(0x1f4ca)} Upload your input CSV file with job specifications")
print("Required columns: jobname, seq1_name, seq1_type, seq1")
print("Optional: seq1_copies, seq1_mods, pocket_binder, pocket_contacts, covalent_bonds")
print("Supports up to 10 sequences per job (seq1 through seq10)")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("No CSV file uploaded")

csv_filename = list(uploaded_csv.keys())[0]
print(f"\n{chr(0x2705)} Uploaded: {csv_filename}")

# Setup Google Drive
drive = None
if setup_google_drive:
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        print(f"\n{chr(0x2601)}  Setting up Google Drive...")
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)
        print(f"{chr(0x2705)} Google Drive connected")
    except Exception as e:
        print(f"{chr(0x26a0)}  Google Drive setup failed: {e}")
        drive = None

# ============================================================
# RESOLVE PRE-COMPUTED MSA FOLDER
# ============================================================
import os as _os

def resolve_msa_folder(msa_folder_input, _drive=None):
    """Resolve user's msa_folder input to the actual paired/ directory path."""
    if not msa_folder_input or not msa_folder_input.strip():
        return ''

    name = msa_folder_input.strip().rstrip('/')

    if name.startswith('/'):
        base_path = name
    else:
        base_path = f"/content/{name}"

    if _os.path.isdir(base_path):
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired
        if _os.path.basename(base_path) == 'paired' and _os.listdir(base_path):
            print(f"   MSA folder resolved: {base_path}")
            return base_path

    zip_path = f"{base_path}.zip"
    if not _os.path.isfile(zip_path):
        zip_path = f"/content/{_os.path.basename(name)}.zip"

    if _os.path.isfile(zip_path):
        print(f"   Found local zip: {zip_path}")
        import zipfile as _zf
        extract_to = _os.path.dirname(zip_path) or '/content'
        with _zf.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        print(f"   Unzipped to {extract_to}")
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired

    if _drive:
        zip_filename = f"{_os.path.basename(name)}.zip"
        print(f"   Searching Google Drive for {zip_filename}...")
        try:
            file_list = _drive.ListFile({
                'q': f"title='{zip_filename}' and trashed=false"
            }).GetList()
            if file_list:
                gdrive_file = file_list[0]
                local_zip = f"/content/{zip_filename}"
                print(f"   Downloading {zip_filename} from Google Drive...")
                gdrive_file.GetContentFile(local_zip)
                print(f"   Downloaded ({_os.path.getsize(local_zip) / 1024 / 1024:.1f} MB)")

                import zipfile as _zf
                with _zf.ZipFile(local_zip, 'r') as zf:
                    zf.extractall('/content')
                print(f"   Unzipped to /content/")

                paired = _os.path.join(base_path, 'paired')
                if _os.path.isdir(paired) and _os.listdir(paired):
                    print(f"   MSA folder resolved: {paired}")
                    return paired
            else:
                print(f"   {zip_filename} not found on Google Drive")
        except Exception as e:
            print(f"   Google Drive download failed: {e}")

    print(f"   WARNING: Could not resolve MSA folder '{msa_folder_input}'")
    return ''

resolved_msa_folder = ''
if msa_folder:
    print(f"\nResolving MSA folder: {msa_folder}")
    resolved_msa_folder = resolve_msa_folder(msa_folder, drive)
    if resolved_msa_folder:
        print(f"   Using pre-computed MSAs from: {resolved_msa_folder}")
        msa_files = [f for f in _os.listdir(resolved_msa_folder) if f.endswith('_paired.a3m')]
        print(f"   Found {len(msa_files)} paired MSA files")

# Store settings (processor and jobs added in Cell 4)
global_settings = {
    'csv_filename': csv_filename,
    'drive': drive,
    'gdrive_folder_name': gdrive_folder_name,
    'msa_folder': resolved_msa_folder,
}

print("\n" + "=" * 60)
print(f"{chr(0x2705)} CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + predict.")
print("=" * 60)

In [ ]:
#@title Cell 3: Install Boltz-2
#@markdown Installs Boltz-2 with pinned dependencies. Safe to re-run -- skips if already installed.
%%time

import subprocess
import sys
import os
import re
import time

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"[{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {result.stderr[:300]}")
        return False
    print("OK")
    return True

def get_cuda_version():
    """Detect CUDA version from nvidia-smi"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                version = match.group(1)
                major = int(version.split('.')[0])
                minor = int(version.split('.')[1])
                return major, minor, version
    except Exception as e:
        print(f"{chr(0x26a0)}  Could not detect CUDA version: {e}")
    return None, None, None

def test_cuequivariance_kernels():
    """Test if cuEquivariance triangle kernels are available"""
    print("\n" + "=" * 60)
    print("CUEQUIVARIANCE KERNEL PREFLIGHT TEST")
    print("=" * 60)

    try:
        import torch
        print(f"{chr(0x2705)} PyTorch: {torch.__version__}")
        print(f"{chr(0x2705)} CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"{chr(0x2705)} CUDA version: {torch.version.cuda}")
            print(f"{chr(0x2705)} GPU: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"{chr(0x274c)} PyTorch check failed: {e}")
        return False

    # Test cuequivariance-torch import
    try:
        import cuequivariance_torch
        print(f"{chr(0x2705)} cuequivariance-torch installed")
    except ImportError as e:
        print(f"{chr(0x26a0)}  cuequivariance-torch not found: {e}")
        return False

    # Test cuequivariance-ops-torch-cu12 import
    try:
        import cuequivariance_ops_torch
        print(f"{chr(0x2705)} cuequivariance-ops-torch-cu12 installed")
    except ImportError as e:
        print(f"{chr(0x26a0)}  cuequivariance-ops-torch-cu12 not found: {e}")
        return False

    # CRITICAL TEST: triangle_multiplicative_update
    try:
        from cuequivariance_ops_torch.triangle import triangle_multiplicative_update
        print(f"{chr(0x2705)} triangle_multiplicative_update import: SUCCESS")

        if callable(triangle_multiplicative_update):
            print(f"{chr(0x2705)} triangle_multiplicative_update is callable")
        else:
            print(f"{chr(0x274c)} triangle_multiplicative_update exists but is not callable")
            return False

    except ImportError as e:
        print(f"{chr(0x274c)} triangle_multiplicative_update import FAILED: {e}")
        print(f"   This error requires --no_kernels flag")
        return False
    except Exception as e:
        print(f"{chr(0x274c)} Unexpected error testing triangle kernels: {e}")
        return False

    # Test triangle_attention_update
    try:
        from cuequivariance_ops_torch.triangle import triangle_attention_update
        print(f"{chr(0x2705)} triangle_attention_update import: SUCCESS")
    except ImportError as e:
        print(f"{chr(0x26a0)}  triangle_attention_update import failed: {e}")
        return False

    print("=" * 60)
    return True


restart_marker = "/content/.boltz_numpy_restart"
# POST-RESTART: INSTALLATION
print("=" * 60)
print("INSTALLING BOLTZ-2")
print("=" * 60)

# Detect CUDA version
cuda_major, cuda_minor, cuda_version = get_cuda_version()

if cuda_major is None:
    print(f"{chr(0x274c)} No CUDA detected - cannot proceed")
    sys.exit(1)

print(f"{chr(0x2705)} CUDA Version: {cuda_version}")
print(f"   Driver CUDA: {cuda_major}.{cuda_minor}")

# PyTorch version (pip resolves CUDA-compatible wheel automatically)
pytorch_version = "2.9.0"
print(f"\n{chr(0x1f4e6)} PyTorch: {pytorch_version} (pip resolves CUDA wheel automatically)")

# Clean existing installations
print("\n" + "=" * 60)
print("[Cleanup]")
cleanup_packages = [
    'torch', 'torchvision', 'torchaudio',
    'pytorch-lightning', 'torchmetrics',
    'boltz', 'numpy', 'pandas'
]
subprocess.run(
    f"{sys.executable} -m pip uninstall {' '.join(cleanup_packages)} -y",
    shell=True, capture_output=True
)
print("OK")

# Install NumPy FIRST
print("\n" + "=" * 60)
print("[1/5] NumPy 1.26.4 - FIRST (Colab compatibility)")
if not run_cmd(
    f"{sys.executable} -m pip install --no-cache-dir 'numpy==1.26.4'",
    "Installing numpy==1.26.4"
):
    print(f"{chr(0x274c)} NumPy installation failed")
    sys.exit(1)

# Verify NumPy
result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
)
if '1.26' not in result.stdout:
    print(f"{chr(0x274c)} NumPy version wrong: {result.stdout.strip()}")
    sys.exit(1)
print(f"   {chr(0x2705)} Verified: NumPy {result.stdout.strip()}")

# Install PyTorch
print("\n" + "=" * 60)
print(f"[2/5] PyTorch {pytorch_version}")
if not run_cmd(
    f"{sys.executable} -m pip install torch=={pytorch_version} torchvision torchaudio",
    f"Installing PyTorch {pytorch_version}"
):
    print(f"{chr(0x274c)} PyTorch installation failed")
    sys.exit(1)

# Install pytorch-lightning
print("\n" + "=" * 60)
print("[3/5] Lightning stack")
if not run_cmd(
    f"{sys.executable} -m pip install pytorch-lightning==2.5.0 torchmetrics==1.4.0",
    "Installing Lightning"
):
    print(f"{chr(0x274c)} Lightning installation failed")
    sys.exit(1)

# Install Boltz
print("\n" + "=" * 60)
print("[4/5] Boltz-2 (+ dependencies)")
print("   Note: cuequivariance packages managed by Boltz")
if not run_cmd(
    f"{sys.executable} -m pip install 'boltz[cuda]'",
    "Installing Boltz-2 (with CUDA kernels)"
):
    print(f"{chr(0x274c)} Boltz installation failed")
    sys.exit(1)

# ── Blackwell GPU (sm_120) compatibility patches ──
# On Blackwell GPUs:
#   - cuEquivariance's pynvml detection fails → patch fallback defaults
#   - Triton-based triangle_multiplicative segfaults → gate on env var
#   - triangle_attention CUDA cubins work fine at full speed
try:
    _cap_r = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    _compute_cap = float(_cap_r.stdout.strip().splitlines()[0]) if _cap_r.returncode == 0 else 0
    if _compute_cap >= 12.0:
        print(chr(10) + '=' * 60)
        print('BLACKWELL GPU DETECTED (compute capability ' + str(_compute_cap) + ')')
        print('Applying Boltz-2 Blackwell compatibility patches...')
        print('=' * 60)
        import site as _site, glob as _glob
        _sp = _site.getsitepackages()[0]

        # Patch 1: cuEquivariance GPU detection fallback defaults
        _cueq_files = _glob.glob(
            os.path.join(_sp, '**', 'cuequivariance_ops', 'triton', 'cache_manager.py'),
            recursive=True
        )
        if _cueq_files:
            _cueq_file = _cueq_files[0]
            with open(_cueq_file) as _f:
                _src = _f.read()
            _cueq_replacements = {
                '"NVIDIA RTX A6000"': '"Generic High-End GPU"',
                '"major": 8': '"major": 12',
                '"minor": 6': '"minor": 0',
                '"total_memory": 45': '"total_memory": 96',
                '"multi_processor_count": 84': '"multi_processor_count": 188',
                '"power_limit": 300': '"power_limit": 600',
                '"clock_rate": 2100': '"clock_rate": 3090',
            }
            _changed = False
            for _old, _new in _cueq_replacements.items():
                if _old in _src:
                    _src = _src.replace(_old, _new)
                    _changed = True
            if _changed:
                with open(_cueq_file, 'w') as _f:
                    _f.write(_src)
                print(f'   {chr(0x2705)} Patch 1: cuEquivariance fallback defaults updated')
            else:
                print(f'   {chr(0x2705)} Patch 1: cuEquivariance fallback already patched')
        else:
            print(f'   {chr(0x26a0)}  cuEquivariance not installed, skipping Patch 1')

        # Patch 2: Gate trimul kernel on BOLTZ_DISABLE_TRIMUL_KERNEL env var
        _trimul_file = os.path.join(_sp, 'boltz/model/layers/triangular_mult.py')
        if os.path.exists(_trimul_file):
            with open(_trimul_file) as _f:
                _src = _f.read()
            if 'import os' not in _src:
                _lines = _src.split(chr(10))
                _last_import = 0
                for _i, _line in enumerate(_lines):
                    if _line.startswith('import ') or _line.startswith('from '):
                        _last_import = _i
                _lines.insert(_last_import + 1, 'import os')
                _src = chr(10).join(_lines)
            _old_check = 'if use_kernels:'
            _new_check = 'if use_kernels and not os.environ.get("BOLTZ_DISABLE_TRIMUL_KERNEL"):'
            _count = _src.count(_old_check)
            if _count > 0:
                _src = _src.replace(_old_check, _new_check)
                with open(_trimul_file, 'w') as _f:
                    _f.write(_src)
                print(f'   {chr(0x2705)} Patch 2: Gated {_count} trimul kernel checks on env var')
            else:
                print(f'   {chr(0x26a0)}  Patch 2: "if use_kernels:" not found — may already be patched')
        else:
            print(f'   {chr(0x26a0)}  {_trimul_file} not found — Patch 2 skipped')

        if 'global_settings' not in globals():
            global_settings = {}
        global_settings['is_blackwell'] = True
        print(chr(10) + '   Blackwell mixed-mode: triatt=cuEquivariance (full speed), trimul=PyTorch fallback')
    else:
        if 'global_settings' not in globals():
            global_settings = {}
        global_settings['is_blackwell'] = False
except Exception as _e:
    print(f'   {chr(0x26a0)}  Blackwell patches skipped: {_e}')
    if 'global_settings' not in globals():
        global_settings = {}
    global_settings['is_blackwell'] = False

# PERMANENT FIX: Create sitecustomize.py
print("\n" + "=" * 60)
print("INSTALLING PERMANENT SYS.PATH FIX")
print("=" * 60)

sitecustomize_content = """# Colab Feb 2025 NumPy priority fix
import sys
import os

if '/env/python' in sys.path:
sys.path.remove('/env/python')

if 'PYTHONPATH' in os.environ:
del os.environ['PYTHONPATH']
"""

sitecustomize_path = "/usr/local/lib/python3.12/dist-packages/sitecustomize.py"
with open(sitecustomize_path, "w") as f:
    f.write(sitecustomize_content)

print(f"   {chr(0x2705)} Created {sitecustomize_path}")
print("   This fixes sys.path on EVERY future kernel start")

# IPython startup script
ipython_startup_dir = "/root/.ipython/profile_default/startup"
os.makedirs(ipython_startup_dir, exist_ok=True)

ipython_fix_path = os.path.join(ipython_startup_dir, "00-fix_syspath.py")
with open(ipython_fix_path, "w") as f:
    f.write(sitecustomize_content)

print(f"   {chr(0x2705)} Created {ipython_fix_path}")
print("   Backup fix for IPython kernels")

# APPLY FIX TO CURRENT KERNEL
print("\n" + "=" * 60)
print("APPLYING FIX TO CURRENT KERNEL")
print("=" * 60)

if '/env/python' in sys.path:
    sys.path.remove('/env/python')
    print(f"   {chr(0x2705)} Removed /env/python from sys.path")
else:
    print(f"   {chr(0x2705)} /env/python not in sys.path")

if 'PYTHONPATH' in os.environ:
    del os.environ['PYTHONPATH']
    print(f"   {chr(0x2705)} Cleared PYTHONPATH environment variable")

# Clear cached imports
modules_to_clear = [key for key in list(sys.modules.keys())
                   if key.startswith(('numpy', 'pandas', 'np', 'pd'))]
for mod in modules_to_clear:
    del sys.modules[mod]

if modules_to_clear:
    print(f"   {chr(0x2705)} Cleared {len(modules_to_clear)} cached modules")

# Step 5: Install ipSAE_batch for interface analysis
print("\n" + "=" * 60)
print("[5/5] ipSAE_batch (interface scoring and visualization)")
run_cmd(
    f"{sys.executable} -m pip install seaborn pycirclize python-igraph plotly",
    "Installing visualization dependencies"
)
if not run_cmd(
    f"{sys.executable} -m pip install git+https://github.com/JKourelis/ipSAE_batch.git",
    "Installing ipSAE_batch"
):
    print("WARNING: ipSAE_batch failed to install. Interface analysis will be unavailable.")

# VERIFICATION IN CURRENT KERNEL
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

print("\n[Testing imports in current kernel]")
try:
    import numpy as np
    import pandas as pd

    print(f"   {chr(0x2705)} NumPy {np.__version__}")
    print(f"   {chr(0x2705)} Pandas {pd.__version__}")

    if not np.__version__.startswith('1.26'):
        print(f"   {chr(0x274c)} NumPy version wrong: expected 1.26.x, got {np.__version__}")
        sys.exit(1)

    print(f"\n   {chr(0x1f389)} All imports working!")

except Exception as e:
    print(f"   {chr(0x274c)} Import failed: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# CUEQUIVARIANCE KERNEL TEST
if global_settings.get('is_blackwell', False):
    # Blackwell: cuEquivariance triatt works (CUDA cubins), but trimul segfaults (Triton).
    # Patches in install cell gate trimul on env var. But first check if cuEquivariance
    # is installed and functional on this GPU architecture.
    try:
        import cuequivariance_ops_torch
        print(f"\n{chr(0x2699)}  Blackwell: cuEquivariance available, using mixed kernel mode")
        print("   triangle_attention: cuEquivariance CUDA cubins (full speed)")
        print("   triangle_multiplicative: PyTorch native (Triton incompatible with sm_120)")
        use_no_kernels = False
    except ImportError:
        print(f"\n{chr(0x2699)}  Blackwell: cuEquivariance not installed, using --no_kernels")
        print("   Install boltz[cuda] for partial kernel acceleration on Blackwell")
        use_no_kernels = True
else:
    kernels_available = test_cuequivariance_kernels()

    if kernels_available:
        print(f"\n{chr(0x2705)} KERNEL TEST PASSED")
        print("   cuEquivariance kernels are available")
        print("   Will run Boltz-2 WITHOUT --no_kernels flag")
        use_no_kernels = False
    else:
        print(f"\n{chr(0x274c)} KERNEL TEST FAILED")
        print("   cuEquivariance kernels are NOT available")
        print("   Will run Boltz-2 WITH --no_kernels flag")
        print("   Performance penalty: ~12 seconds per prediction")
        use_no_kernels = True

# Store result for execution cell
if 'global_settings' not in globals():
    global_settings = {}
global_settings['use_no_kernels'] = use_no_kernels
global_settings['kernels_tested'] = True

print(f"\n{chr(0x1f527)} Flag stored: use_no_kernels = {use_no_kernels}")

# Test ipSAE_batch
result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("   ipsae-batch CLI: OK")
else:
    print("   ipsae-batch CLI: NOT AVAILABLE (interface analysis will be skipped)")

# ============================================================
# ENVIRONMENT & VERSION LOG
# ============================================================
env_lines = []
env_lines.append("\n" + "=" * 60)
env_lines.append("ENVIRONMENT & INSTALLED VERSIONS")
env_lines.append("=" * 60)

# Python version
env_lines.append(f"\nPython: {sys.version.split()[0]}")

# GPU info
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu_result.returncode == 0:
    env_lines.append(f"GPU: {gpu_result.stdout.strip()}")
env_lines.append(f"CUDA (driver max): {cuda_version}")

# PyTorch + CUDA runtime
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'PyTorch: {torch.__version__}'); "
     "print(f'CUDA available: {torch.cuda.is_available()}'); "
     "print(f'CUDA runtime: {torch.version.cuda}') if torch.cuda.is_available() else None"],
    capture_output=True, text=True
)
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        env_lines.append(f"   {line}")

# All relevant package versions via pip
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze"],
    capture_output=True, text=True
)
all_packages = result.stdout.strip().split('\n')

# Boltz-2 + core deps
env_lines.append("\nBoltz-2 stack:")
tool_pkgs = [
    'boltz',
    'numpy',
    'torch',
    'torchvision',
    'torchaudio',
    'pytorch-lightning',
    'hydra-core',
    'rdkit',
    'dm-tree',
    'einops',
    'einx',
    'fairscale',
    'mashumaro',
    'modelcif',
    'wandb',
    'biopython',
    'scipy',
    'numba',
    'gemmi',
    'scikit-learn',
    'pandas',
    'cuequivariance-torch',
    'cuequivariance-ops-torch-cu12',
]
for pkg in tool_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        # Try partial match
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break

# ipSAE_batch + visualization deps
env_lines.append("\nipSAE_batch stack:")
ipsae_pkgs = [
    'ipsae-batch', 'seaborn', 'pycirclize', 'python-igraph', 'plotly',
    'matplotlib', 'networkx',
]
for pkg in ipsae_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break
        else:
            env_lines.append(f"   {pkg}: not installed")


# Cleanup and mark ready
os.remove(restart_marker)
with open("/content/BOLTZ_READY", "w") as f:
    f.write("Ready")

env_lines.append("\n" + "=" * 60)

# Write environment log to file and print
env_log = "\n".join(str(x) for x in env_lines)
print(env_log)
with open("/content/environment.txt", "w") as _ef:
    _ef.write(env_log + "\n")
print(f"{chr(0x2705)} BOLTZ-2 INSTALLATION COMPLETE")
print("=" * 60)
print("Next: Run Cell 3 to set up CSV processor")


In [ ]:
#@title Cell 4: Boltz-2 CSV Processor Setup
import pandas as pd
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO

class BoltzJobProcessor:
    """Process CSV files for Boltz-2 batch predictions"""

    EMBEDDED_REFERENCE = """Type,CCD Code,Name,Target Residue,Molecular Formula,All Atom Count,Heavy Atom Count,SMILES
PTM,SEP,Phosphoserine,SER,C3H8NO6P,18,11
PTM,TPO,Phosphothreonine,THR,C4H10NO6P,21,12
PTM,PTR,Phosphotyrosine,TYR,C9H12NO6P,28,17
PTM,MLY,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,ALY,N-Acetyllysine,LYS,C8H16N2O3,29,13
PTM,HYP,Hydroxyproline,PRO,C5H9NO3,18,9
PTM,M3L,N-Trimethyllysine,LYS,C9H20N2O2,33,13
PTM,MLZ,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,CSD,Cysteine sulfinic acid,CYS,C3H7NO4S,16,9
PTM,CSO,S-Hydroxycysteine,CYS,C3H7NO3S,15,8
PTM,CGU,Gamma-carboxyglutamic acid,GLU,C5H7NO6,21,12
PTM,FME,N-Formylmethionine,MET,C6H11NO3S,22,11
PTM,NEP,N-(phosphonoethyl)isoleucine,ILE,C8H18NO5P,32,15
PTM,HIC,4-Methyl-histidine,HIS,C7H11N3O2,23,12
PTM,CAS,S-(dimethylarsenic)cysteine,CYS,C5H11AsNO2S,20,9
Ligand,ATP,Adenosine triphosphate,NA,C10H16N5O13P3,47,31,Nc1ncnc2n(cnc12)[C@@H]1O[C@H](COP(O)(=O)OP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,ADP,Adenosine diphosphate,NA,C10H15N5O10P2,42,27,Nc1ncnc2n(cnc12)[C@@H]1O[C@H](COP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,AMP,Adenosine monophosphate,NA,C10H14N5O7P,37,23,Nc1ncnc2n(cnc12)[C@@H]1O[C@H](COP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,GTP,Guanosine triphosphate,NA,C10H16N5O14P3,48,32,NC1=Nc2n(cnc2C(=O)N1)[C@@H]1O[C@H](COP(O)(=O)OP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,GDP,Guanosine diphosphate,NA,C10H15N5O11P2,43,28,NC1=Nc2n(cnc2C(=O)N1)[C@@H]1O[C@H](COP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,GMP,Guanosine monophosphate,NA,C10H14N5O8P,38,24,NC1=Nc2n(cnc2C(=O)N1)[C@@H]1O[C@H](COP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,CTP,Cytidine triphosphate,NA,C9H16N3O14P3,45,29,NC1=NC(=O)N(C=C1)[C@@H]1O[C@H](COP(O)(=O)OP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,CDP,Cytidine diphosphate,NA,C9H15N3O11P2,40,25,NC1=NC(=O)N(C=C1)[C@@H]1O[C@H](COP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,UTP,Uridine triphosphate,NA,C9H15N2O15P3,44,29,O=C1NC(=O)C=CN1[C@@H]1O[C@H](COP(O)(=O)OP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,UDP,Uridine diphosphate,NA,C9H14N2O12P2,39,25,O=C1NC(=O)C=CN1[C@@H]1O[C@H](COP(O)(=O)OP(O)(O)=O)[C@@H](O)[C@H]1O
Ligand,NAD,Nicotinamide adenine dinucleotide,NA,C21H27N7O14P2,71,44,NC(=O)c1ccc[n+](c1)[C@@H]1O[C@H](COP([O-])(=O)OP(O)(=O)OC[C@H]2O[C@H]([C@H](O)[C@@H]2O)n2cnc3c(N)ncnc23)[C@@H](O)[C@H]1O
Ligand,NAP,NADP,NA,C21H28N7O17P3,86,55,NC(=O)c1ccc[n+](c1)[C@@H]1O[C@H](COP([O-])(=O)OP(O)(=O)OC[C@H]2O[C@H]([C@H](OP(O)(O)=O)[C@@H]2O)n2cnc3c(N)ncnc23)[C@@H](O)[C@H]1O
Ligand,FAD,Flavin adenine dinucleotide,NA,C27H33N9O15P2,91,53,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)COP(O)(=O)OP(O)(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n3cnc4c(N)ncnc34)c2cc1C
Ligand,FMN,Flavin mononucleotide,NA,C17H21N4O9P,52,31,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)COP(O)(O)=O)c2cc1C
Ligand,HEM,Heme,NA,C34H32FeN4O4,75,43,CC1=C(CCC(O)=O)C2=Cc3c(C)c(CCC(O)=O)c4C=C5C(C)=C(C=C)C6=[N+]5[Fe--]5(n34)n3c(=C1[N+]2=5)c(C)c3C=C6
Ligand,SAH,S-Adenosyl-L-homocysteine,NA,C14H20N6O5S,46,26,N[C@@H](CCSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n1cnc2c(N)ncnc12)C(O)=O
Ligand,SAM,S-Adenosyl-L-methionine,NA,C15H22N6O5S,49,27,C[S@@+](CC[C@H](N)C([O-])=O)C[C@H]1O[C@H]([C@H](O)[C@@H]1O)n1cnc2c(N)ncnc12
Ligand,COA,Coenzyme A,NA,C21H36N7O16P3S,90,57,CC(C)(COP(O)(=O)OP(O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1OP(O)(O)=O)n1cnc2c(N)ncnc12)[C@@H](O)C(=O)NCCC(=O)NCCS
Ligand,ACO,Acetyl coenzyme A,NA,C23H38N7O17P3S,99,61,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(O)(=O)OP(O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1OP(O)(O)=O)n1cnc2c(N)ncnc12
Ligand,PLP,Pyridoxal-5-phosphate,NA,C8H10NO6P,25,16,Cc1ncc(COP(O)(O)=O)c(C=O)c1O
Ligand,TPP,Thiamine diphosphate,NA,C12H19N4O7P2S,45,25,Cc1ncc(C[n+]2csc(CCOP(O)(=O)OP(O)(O)=O)c2C)c(N)n1
Ligand,BTN,Biotin,NA,C10H16N2O3S,32,16,OC(=O)CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@H]12
Ligand,MTA,5-Methylthioadenosine,NA,C11H15N5O3S,35,20,CSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n1cnc2c(N)ncnc12
Ligand,THM,Thiamine,NA,C12H17ClN4OS,38,18,Cc1ncc(C[n+]2csc(CCO)c2C)c(N)n1
Ion,MG,Magnesium ion,NA,Mg,1,1
Ion,ZN,Zinc ion,NA,Zn,1,1
Ion,CA,Calcium ion,NA,Ca,1,1
Ion,FE,Iron ion,NA,Fe,1,1
Ion,MN,Manganese ion,NA,Mn,1,1
Ion,CU,Copper ion,NA,Cu,1,1
Ion,CO,Cobalt ion,NA,Co,1,1
Ion,NI,Nickel ion,NA,Ni,1,1
Ion,K,Potassium ion,NA,K,1,1
Ion,NA,Sodium ion,NA,Na,1,1
Ion,CL,Chloride ion,NA,Cl,1,1
Glycan,NAG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,MAN,D-Mannose,NA,C6H12O6,24,12
Glycan,FUC,L-Fucose,NA,C6H12O5,23,11
Glycan,GAL,D-Galactose,NA,C6H12O6,24,12
Glycan,SIA,N-Acetylneuraminic acid,NA,C11H19NO9,40,21
Glycan,GLC,D-Glucose,NA,C6H12O6,24,12
Glycan,BMA,beta-D-Mannose,NA,C6H12O6,24,12
Glycan,NDG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,A2G,N-Acetyl-D-galactosamine,NA,C8H15NO6,30,15
Glycan,FUL,L-Fucose,NA,C6H12O5,23,11
DNA_Mod,5MC,5-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,6MA,N6-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,5HMC,5-Hydroxymethylcytosine,DC,C10H15N3O8P,37,22
DNA_Mod,8OG,8-Oxoguanine,DG,C10H13N5O8P,37,24
DNA_Mod,M2G,N2-Methylguanine,DG,C11H16N5O7P,40,24
DNA_Mod,4MC,N4-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,1MA,1-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,3MA,3-Methyladenine,DA,C11H16N5O6P,39,23
RNA_Mod,PSU,Pseudouridine,U,C9H12N2O9P,33,21
RNA_Mod,1MA,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,7MG,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,5MC,5-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,2MA,2-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,M2G,N2-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M7G,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M1A,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,OMC,2'-O-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,OMG,2'-O-Methylguanosine,G,C11H15N5O8P,40,25"""

    def __init__(self, reference_csv: Optional[str] = None):
        """Initialize processor with reference data"""
        if reference_csv:
            self.reference_data = pd.read_csv(StringIO(reference_csv))
        else:
            self.reference_data = pd.read_csv(StringIO(self.EMBEDDED_REFERENCE))

        self.ptm_lookup = self._create_lookup('PTM')
        self.ligand_lookup = self._create_lookup('Ligand')
        self.ion_lookup = self._create_lookup('Ion')
        self.glycan_lookup = self._create_lookup('Glycan')
        self.dna_mod_lookup = self._create_lookup('DNA_Mod')
        self.rna_mod_lookup = self._create_lookup('RNA_Mod')

        self.aa_3to1 = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
        }

        self.amino_acids = set('ACDEFGHIKLMNPQRSTVWY')

    def _create_lookup(self, type_name: str) -> Dict[str, Dict[str, Any]]:
        """Create lookup dictionary for a specific type"""
        type_data = self.reference_data[self.reference_data['Type'] == type_name]
        lookup = {}
        for _, row in type_data.iterrows():
            if pd.notna(row['CCD Code']):
                lookup[row['CCD Code']] = {
                    'name': row['Name'],
                    'target_residue': row['Target Residue'] if pd.notna(row['Target Residue']) else 'NA',
                    'smiles': row['SMILES'] if 'SMILES' in row.index and pd.notna(row.get('SMILES')) else None
                }
        return lookup

    def _validate_sequence_characters(self, sequence: str, seq_type: str) -> List[str]:
        """Validate sequence contains only allowed characters"""
        errors = []
        sequence = sequence.upper()

        if seq_type.lower() == 'protein':
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in self.amino_acids:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid amino acids: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'dna':
            valid_bases = set('ATCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid DNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'rna':
            valid_bases = set('AUCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid RNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        return errors

    def _is_smiles(self, ligand_string: str) -> bool:
        """Check if string is likely a SMILES representation"""
        if len(ligand_string) < 3:
            return False
        smiles_chars = set('[]()=#@+-\\/CNOPSFClBrI0123456789')
        return any(char in smiles_chars for char in ligand_string)

    def _remap_modification_chains(self, mod_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap modification chain IDs from user names to A, B, C... format"""
        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mod_string

        mod_string = str(mod_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            mod_string = mod_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return mod_string

    def _remap_contact_chains(self, contacts_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap contact chain IDs from user names to A, B, C... format"""
        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            return contacts_string

        contacts_string = str(contacts_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            contacts_string = contacts_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return contacts_string

    def _remap_bond_chains(self, bonds_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap covalent bond chain IDs from user names to A, B, C... format"""
        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds_string

        bonds_string = str(bonds_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            bonds_string = bonds_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return bonds_string

    def _parse_modifications(self, mod_string: str, sequence: str, chain_id: str, seq_type: str) -> Tuple[List[Dict], List[str]]:
        """Parse modifications in format: CHAIN:POSITION:CCD_CODE"""
        errors = []
        mods = []

        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mods, errors

        mod_string = str(mod_string).strip()
        mod_entries = [e.strip() for e in mod_string.split(';') if e.strip()]

        if seq_type.lower() == 'protein':
            mod_lookups = [self.ptm_lookup, self.glycan_lookup]
            mod_types = ['PTM', 'Glycan']
        elif seq_type.lower() == 'dna':
            mod_lookups = [self.dna_mod_lookup]
            mod_types = ['DNA Mod']
        elif seq_type.lower() == 'rna':
            mod_lookups = [self.rna_mod_lookup]
            mod_types = ['RNA Mod']
        else:
            return mods, errors

        for entry in mod_entries:
            parts = entry.split(':')
            if len(parts) != 3:
                errors.append(f"Invalid modification format: '{entry}'. Use CHAIN:POSITION:CCD_CODE")
                continue

            mod_chain, pos_str, ccd_code = parts
            mod_chain = mod_chain.strip()
            ccd_code = ccd_code.strip()

            if mod_chain != chain_id:
                continue

            try:
                position = int(pos_str.strip())

                found = False
                for lookup, mod_type in zip(mod_lookups, mod_types):
                    if ccd_code in lookup:
                        found = True

                        if position < 1 or position > len(sequence):
                            errors.append(f"{mod_type} position {position} out of range (sequence length: {len(sequence)})")
                            continue

                        target_residue = lookup[ccd_code]['target_residue']
                        actual_residue = sequence[position - 1].upper()

                        if target_residue != 'NA':
                            if mod_type == 'Glycan':
                                if actual_residue not in ['N', 'T', 'S']:
                                    errors.append(f"Glycan {ccd_code} requires N/T/S but found {actual_residue} at position {position}")
                                    continue
                            elif mod_type == 'PTM':
                                target_1letter = self.aa_3to1.get(target_residue.upper())
                                if target_1letter and actual_residue != target_1letter:
                                    errors.append(f"PTM {ccd_code} targets {target_residue}({target_1letter}) but found {actual_residue} at position {position}")
                                    continue
                            else:
                                if len(target_residue) > 1 and target_residue.startswith('D'):
                                    target_residue = target_residue[1]
                                if actual_residue != target_residue:
                                    errors.append(f"{mod_type} {ccd_code} targets {target_residue} but found {actual_residue} at position {position}")
                                    continue

                        mods.append({
                            'chain_id': mod_chain,
                            'position': position,
                            'ccd': ccd_code
                        })
                        break

                if not found:
                    errors.append(f"Unknown modification code: '{ccd_code}'")

            except ValueError:
                errors.append(f"Invalid position in modification: '{entry}'")

        return mods, errors

    def _parse_pocket(self, binder: str, contacts_string: str) -> Tuple[Optional[Dict], List[str]]:
        """Parse pocket constraint"""
        errors = []

        if pd.isna(binder) or str(binder).strip() == '':
            return None, errors

        binder = str(binder).strip()

        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            errors.append(f"Pocket binder '{binder}' specified but no contacts provided")
            return None, errors

        contacts_string = str(contacts_string).strip()
        contact_entries = [e.strip() for e in contacts_string.split(';') if e.strip()]

        contacts = []
        for entry in contact_entries:
            parts = entry.split(':')
            if len(parts) != 2:
                errors.append(f"Invalid contact format: '{entry}'. Use CHAIN:RESIDUE")
                continue

            try:
                chain_id, residue_num = parts
                contacts.append([chain_id.strip(), int(residue_num.strip())])
            except ValueError:
                errors.append(f"Invalid contact specification: '{entry}'")

        if not contacts:
            errors.append(f"No valid contacts parsed for pocket binder '{binder}'")
            return None, errors

        return {
            'binder': binder,
            'contacts': contacts
        }, errors

    def _parse_covalent_bonds(self, bonds_string: str) -> Tuple[List[Dict], List[str]]:
        """Parse covalent bond constraints"""
        errors = []
        bonds = []

        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds, errors

        bonds_string = str(bonds_string).strip()
        bond_entries = [e.strip() for e in bonds_string.split(';') if e.strip()]

        for entry in bond_entries:
            parts = entry.split(':')
            if len(parts) != 6:
                errors.append(f"Invalid covalent bond format: '{entry}'. Use CHAIN:RES:ATOM:CHAIN:RES:ATOM")
                continue

            try:
                chain1, res1, atom1, chain2, res2, atom2 = parts
                bonds.append({
                    'atom1': [chain1.strip(), int(res1.strip()), atom1.strip()],
                    'atom2': [chain2.strip(), int(res2.strip()), atom2.strip()]
                })
            except ValueError:
                errors.append(f"Invalid covalent bond specification: '{entry}'")

        return bonds, errors

    def _sanitize_jobname(self, jobname: str) -> Tuple[str, List[str]]:
        """Sanitize jobname for filesystem compatibility"""
        errors = []

        if pd.isna(jobname) or str(jobname).strip() == '':
            errors.append("Missing jobname")
            return '', errors

        original = str(jobname)
        sanitized = original.lower()
        sanitized = sanitized.replace('-', '_')
        sanitized = re.sub(r'[^a-z0-9_]', '', sanitized)

        if len(sanitized) > 128:
            sanitized = sanitized[:128]
            errors.append(f"Jobname truncated to 128 characters")

        if not sanitized.strip():
            errors.append(f"Jobname '{original}' became empty after sanitization")
            return '', errors

        return sanitized, errors

    def _generate_yaml(self, job: Dict) -> str:
        """Generate YAML format for Boltz-2"""
        lines = ["version: 1", "sequences:"]

        protein_groups = {}
        dna_groups = {}
        rna_groups = {}
        ligand_groups = {}

        for seq in job['sequences']:
            seq_type = seq['type']

            if seq_type == 'protein':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in protein_groups:
                    protein_groups[key] = []
                protein_groups[key].append(seq)

            elif seq_type == 'dna':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in dna_groups:
                    dna_groups[key] = []
                dna_groups[key].append(seq)

            elif seq_type == 'rna':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in rna_groups:
                    rna_groups[key] = []
                rna_groups[key].append(seq)

            elif seq_type == 'ligand':
                if 'smiles' in seq:
                    key = ('smiles', seq['smiles'])
                else:
                    key = ('ccd', seq['ccd'])
                if key not in ligand_groups:
                    ligand_groups[key] = []
                ligand_groups[key].append(seq)

        for (sequence, mod_tuple), seqs in protein_groups.items():
            lines.append("  - protein:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - ptmType: {mod['ccd']}")
                    lines.append(f"          ptmPosition: {mod['position']}")

        for (sequence, mod_tuple), seqs in dna_groups.items():
            lines.append("  - dna:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - modificationType: {mod['ccd']}")
                    lines.append(f"          basePosition: {mod['position']}")

        for (sequence, mod_tuple), seqs in rna_groups.items():
            lines.append("  - rna:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - modificationType: {mod['ccd']}")
                    lines.append(f"          basePosition: {mod['position']}")

        for (lig_type, lig_value), seqs in ligand_groups.items():
            lines.append("  - ligand:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")

            if lig_type == 'smiles':
                lines.append(f"      smiles: '{lig_value}'")
            else:
                lines.append(f"      ccd: {lig_value}")

        if job.get('pocket') or job.get('covalent_bonds'):
            lines.append("constraints:")

            if job.get('pocket'):
                pocket = job['pocket']
                lines.append("  - pocket:")
                lines.append(f"      binder: {pocket['binder']}")
                lines.append("      contacts:")
                for contact in pocket['contacts']:
                    lines.append(f"        - [{contact[0]}, {contact[1]}]")

            if job.get('covalent_bonds'):
                for bond in job['covalent_bonds']:
                    lines.append("  - bond:")
                    lines.append(f"      atom1: [{bond['atom1'][0]}, {bond['atom1'][1]}, {bond['atom1'][2]}]")
                    lines.append(f"      atom2: [{bond['atom2'][0]}, {bond['atom2'][1]}, {bond['atom2'][2]}]")

        return '\n'.join(lines)

    def _process_job(self, row: pd.Series) -> Tuple[Optional[Dict], List[str]]:
        """Process a single job row from CSV"""
        errors = []
        all_sequences = []

        jobname, jobname_errors = self._sanitize_jobname(row.get('jobname', ''))
        errors.extend(jobname_errors)

        if not jobname:
            return None, errors

        pocket_binder = row.get('pocket_binder', '')
        pocket_contacts = row.get('pocket_contacts', '')
        covalent_bonds_str = row.get('covalent_bonds', '')

        chain_id_counter = 0
        name_to_chain_mapping = {}

        for i in range(1, 11):
            name_col = f'seq{i}_name'
            type_col = f'seq{i}_type'
            copies_col = f'seq{i}_copies'
            seq_col = f'seq{i}'
            mods_col = f'seq{i}_mods'

            if name_col not in row or type_col not in row or seq_col not in row:
                continue

            user_name = row.get(name_col, '')
            seq_type = row.get(type_col, '')
            seq_type = str(seq_type).strip().lower()
            copies = row.get(copies_col, 1)
            sequence = row.get(seq_col, '')
            mods = row.get(mods_col, '')

            if pd.isna(sequence) or str(sequence).strip() == '':
                continue

            sequence = str(sequence).strip()
            copies = int(copies) if pd.notna(copies) and str(copies).strip() != '' else 1
            user_name = str(user_name).strip() if pd.notna(user_name) else ''

            chain_ids = []
            for copy_num in range(copies):
                chain_id = chr(ord('A') + chain_id_counter)
                chain_ids.append(chain_id)

                if user_name and copy_num == 0:
                    name_to_chain_mapping[user_name] = chain_id

                chain_id_counter += 1

            if seq_type.lower() in ['protein', 'dna', 'rna']:
                char_errors = self._validate_sequence_characters(sequence, seq_type)
                errors.extend(char_errors)

                for idx, chain_id in enumerate(chain_ids):
                    remapped_mods = self._remap_modification_chains(mods, name_to_chain_mapping)
                    mods_list, mod_errors = self._parse_modifications(remapped_mods, sequence, chain_id, seq_type)
                    errors.extend(mod_errors)

                    seq_dict = {
                        'type': seq_type.lower(),
                        'id': chain_id,
                        'name': user_name,
                        'sequence': sequence,
                        'modifications': mods_list if mods_list else None
                    }
                    all_sequences.append(seq_dict)

            elif seq_type.lower() == 'ligand':
                ligand_string = sequence.strip()

                # Check CCD lookup FIRST (before SMILES detection)
                # _is_smiles has false positives for CCD codes containing P,S,N,C,O,F,I
                looked_up_smiles = None
                is_known_ccd = False
                if ligand_string in self.ligand_lookup:
                    looked_up_smiles = self.ligand_lookup[ligand_string].get('smiles')
                    is_known_ccd = True
                elif ligand_string in self.ion_lookup:
                    looked_up_smiles = self.ion_lookup[ligand_string].get('smiles')
                    is_known_ccd = True

                if not is_known_ccd and not self._is_smiles(ligand_string):
                    errors.append(f"Unknown ligand/ion: '{ligand_string}'")

                for chain_id in chain_ids:
                    if looked_up_smiles:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': looked_up_smiles
                        }
                    elif is_known_ccd:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'ccd': ligand_string
                        }
                    else:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': ligand_string
                        }
                    all_sequences.append(seq_dict)
            else:
                errors.append(f"Unsupported sequence type: {seq_type}")

        if not all_sequences:
            errors.append("No valid sequences found")
            return None, errors

        remapped_pocket_binder = name_to_chain_mapping.get(pocket_binder, pocket_binder) if pocket_binder else pocket_binder
        remapped_contacts = self._remap_contact_chains(pocket_contacts, name_to_chain_mapping)

        pocket, pocket_errors = self._parse_pocket(remapped_pocket_binder, remapped_contacts)
        errors.extend(pocket_errors)

        remapped_bonds = self._remap_bond_chains(covalent_bonds_str, name_to_chain_mapping)
        covalent_bonds, bond_errors = self._parse_covalent_bonds(remapped_bonds)
        errors.extend(bond_errors)

        has_modifications = any(seq.get('modifications') for seq in all_sequences)

        job = {
            'name': jobname,
            'sequences': all_sequences,
            'pocket': pocket,
            'covalent_bonds': covalent_bonds,
            'has_modifications': has_modifications,
            'has_pocket': pocket is not None,
            'has_covalent': len(covalent_bonds) > 0
        }

        return job, errors

    def process_csv(self, csv_path: str) -> Tuple[List[Dict], pd.DataFrame]:
        """Process CSV file and return jobs list and summary DataFrame"""
        df = pd.read_csv(csv_path)

        jobs = []
        summary_rows = []

        for idx, row in df.iterrows():
            job, errors = self._process_job(row)

            if job:
                jobs.append(job)
                summary_rows.append({
                    'jobname': job['name'],
                    'sequences': len(job['sequences']),
                    'has_modifications': job['has_modifications'],
                    'has_pocket': job['has_pocket'],
                    'has_covalent': job['has_covalent'],
                    'status': 'ERROR' if errors else 'OK',
                    'errors': '; '.join(errors) if errors else ''
                })
            else:
                summary_rows.append({
                    'jobname': f"Row {idx+1}",
                    'sequences': 0,
                    'has_modifications': False,
                    'has_pocket': False,
                    'has_covalent': False,
                    'status': 'FAILED',
                    'errors': '; '.join(errors)
                })

        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

print(f"{chr(0x1f527)} Initializing Boltz CSV Processor...")
boltz_processor = BoltzJobProcessor()

print(f"{chr(0x2705)} Using embedded reference data: 79 entries")
print(f"{chr(0x2705)} Processor ready")
print(f"{chr(0x1f4cb)} Reference data includes:")
print(f"   - 15 PTM types")
print(f"   - 24 ligand types")
print(f"   - 11 ion types")
print(f"   - 10 glycan types")
print(f"   - 8 DNA modification types")
print(f"   - 10 RNA modification types")
print(f"\n{chr(0x1f4a1)} Using embedded reference data (common PTMs, ligands, ions, glycans, DNA/RNA mods)")
print("   To use custom reference: upload file in Cell 3")
print(f"\n{chr(0x1f4dd)} Note: Chain IDs are assigned as A, B, C, D... sequentially")
print("   Sequence identity is preserved in job names")

# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
import pandas as pd

if 'global_settings' not in globals() or 'csv_filename' not in global_settings:
    print("Error: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No CSV uploaded")

csv_filename = global_settings['csv_filename']

# Initialize processor
custom_ref_file = None
try:
    boltz_processor = BoltzJobProcessor(custom_ref_file)
except Exception as e:
    print(f"{chr(0x274c)} Failed to initialize processor: {e}")
    raise

print(f"\n{chr(0x1f504)} Processing CSV...")
try:
    jobs, summary_df = boltz_processor.process_csv(csv_filename)
except Exception as e:
    print(f"{chr(0x274c)} CSV processing failed: {e}")
    raise

print("\n" + "=" * 60)
print(f"{chr(0x1f4cb)} JOB SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

if jobs:
    print("\n" + "=" * 60)
    print("YAML PREVIEW (first job)")
    print("=" * 60)
    preview_yaml = boltz_processor._generate_yaml(jobs[0])
    print(preview_yaml)

error_jobs = summary_df[summary_df['status'] == 'ERROR']
if len(error_jobs) > 0:
    print(f"\n{chr(0x26a0)}  {len(error_jobs)} jobs have errors:")
    for _, row in error_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

    proceed = input("\nProceed with valid jobs only? (yes/no): ").strip().lower()
    if proceed not in ['yes', 'y']:
        raise ValueError("Processing cancelled by user")

global_settings.update({
    'batch_jobs': jobs,
    'processor': boltz_processor,
})

print("\n" + "=" * 60)
print(f"{chr(0x2705)} READY TO PROCESS {len(jobs)} JOBS")
print("=" * 60)
if global_settings.get('msa_folder'):
    print(f"Pre-computed MSAs: {global_settings['msa_folder']}")

print(f"\n{chr(0x1f4cc)} Next: Configure settings (Cell 5), then run (Cell 6)")

In [ ]:
#@title Cell 5: Configure Boltz-2 Prediction Parameters
# ============================================================
# MSA Configuration
# ============================================================
#@markdown > **Note**: Online MSA settings only apply when `msa_folder` is empty (Cell 4).
#@markdown > When pre-computed MSAs are provided, online MSA search is skipped entirely.
#@markdown > `max_msa_seqs` and `subsample_msa` still apply (Boltz-2 processes them at inference time).

msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref", "single_sequence"]
#@markdown - **MSA generation method** (mmseqs2 modes use ColabFold server)

msa_pairing_strategy = "greedy" #@param ["greedy", "complete"]
#@markdown - **Pairing strategy**: `greedy` = pair any matching subsets, `complete` = all sequences must match

max_msa_seqs = 8192 #@param {type:"integer"}
#@markdown - **Max MSA sequences**: Maximum number of MSA sequences to use. Lower values reduce VRAM and speed up inference at cost of accuracy.

subsample_msa = False #@param {type:"boolean"}
#@markdown - **Subsample MSA**: Randomly subsample MSA sequences. Can improve diversity for very deep MSAs.

num_subsampled_msa = 1024 #@param {type:"integer"}
#@markdown - **Subsampled MSA count**: Number of sequences to keep when subsampling. Only used when subsample_msa is True.

#@markdown ---
# ============================================================
# Structure Prediction Settings
# ============================================================
recycling_steps = 6 #@param {type:"integer"}
#@markdown - **Iterative refinement passes**: Each cycle refines the structure using updated predictions. Higher values improve local geometry and confidence scores. **Time**: ~linear scaling. **VRAM**: +20-30% per additional step.

sampling_steps = 200 #@param {type:"integer"}
#@markdown - **Diffusion denoising iterations**: Controls how many steps the diffusion model takes to generate structures from noise. More steps = smoother, higher quality structures. **Time**: Linear scaling. **VRAM**: +10-15%.

diffusion_samples = 5 #@param {type:"integer"}
#@markdown - **Independent structure predictions**: Number of different structures generated per input. **Time**: Linear scaling. **VRAM**: Depends on max_parallel_samples.

max_parallel_samples = 5 #@param {type:"integer"}
#@markdown - **GPU memory management**: How many diffusion samples are processed simultaneously. **Time**: Minimal impact. **VRAM**: ~Linear scaling.

step_scale = 1.5 #@param {type:"number"}
#@markdown - **Sampling temperature**: Controls randomness in structure generation. 1.5 is the Boltz-2 optimized default (1.638 was the Boltz-1 default).

#@markdown ---
# ============================================================
# Affinity Prediction Settings
# ============================================================
predict_affinity = False #@param {type:"boolean"}
#@markdown - **Binding strength prediction**: Runs additional affinity model. **Time**: +50-100%. **VRAM**: +40-60%.

affinity_mw_correction = False #@param {type:"boolean"}
#@markdown - **Molecular weight adjustment**: Applies size-based corrections to affinity predictions.

sampling_steps_affinity = 200 #@param {type:"integer"}
#@markdown - **Affinity model diffusion steps**: Controls quality of affinity predictions.

diffusion_samples_affinity = 5 #@param {type:"integer"}
#@markdown - **Affinity prediction ensemble size**: Number of independent affinity predictions to average.

#@markdown ---
# ============================================================
# Output and Optimization Settings
# ============================================================
output_format = "mmcif" #@param ["mmcif", "pdb"]
#@markdown - **Structure file format**: mmCIF supports more metadata, PDB is more widely compatible.

write_full_pae = True #@param {type:"boolean"}
#@markdown - **Save Predicted Aligned Error matrix**: Essential for assessing interface quality.

write_full_pde = False #@param {type:"boolean"}
#@markdown - **Save Predicted Distance Error matrix**: Distance confidence predictions.

use_potentials = True #@param {type:"boolean"}
#@markdown - **Inference-time physics optimization**: Applies physics-based energy minimization. **Time**: +30-50%. **VRAM**: +15-25%.

# ============================================================
# Reproducibility & Advanced Settings
# ============================================================
seed = "" #@param {type:"string"}
#@markdown - **Random seed**: Integer for reproducible results (e.g., "42"). Leave empty for non-deterministic runs.

write_embeddings = False #@param {type:"boolean"}
#@markdown - **Save embeddings**: Write single/pair representation embeddings to .npz files. Useful for downstream analysis.

skip_existing = False  #@param {type:"boolean"}
#@markdown - **Skip existing**: If True, skip jobs that already have output in their base directory. Useful for resuming interrupted batches.

#@markdown ---
#@markdown ### Interface Analysis (ipSAE_batch)
#@markdown Scores interface quality (ipSAE, ipTM, pDockQ) and generates visualizations for each prediction.

enable_ipsae = True #@param {type:"boolean"}
#@markdown - **Enable**: Run ipSAE interface analysis on each completed job.

ipsae_png = True #@param {type:"boolean"}
#@markdown - **PNG plots**: Matrix + ribbon visualizations per model (~10-30s/job).

ipsae_pdf = False #@param {type:"boolean"}
#@markdown - **PDF report**: Side-by-side model comparison per job (~20-60s/job).

ipsae_per_residue = False #@param {type:"boolean"}
#@markdown - **Per-residue CSV**: Detailed per-residue interface scores.

ipsae_per_contact = False #@param {type:"boolean"}
#@markdown - **Per-contact CSV**: Per-contact-pair scores (most detailed).

ipsae_pae_cutoff = 10.0 #@param {type:"number"}
#@markdown - **PAE cutoff**: PAE threshold for interface scoring (default: 10.0).

ipsae_dist_cutoff = 10.0 #@param {type:"number"}
#@markdown - **Distance cutoff**: CB-CB distance threshold in Angstroms (default: 10.0).

ipsae_select_residues = "" #@param {type:"string"}
#@markdown - **Select residues**: Focus on specific residues, e.g. `A:100-105,B:203`. Empty = all interfaces.

ipsae_batch_analysis = True #@param {type:"boolean"}
#@markdown - **Final batch comparison**: Combined CSV + interactive HTML across all jobs.

# ============================================================
# Apply All Settings
# ============================================================
if 'global_settings' not in globals():
    print(f"{chr(0x26a0)}  Please run the CSV Upload cell first")
else:
    # MSA settings
    if "mmseqs2" in msa_mode:
        use_msa_server = True
        msa_server_url = "https://api.colabfold.com"
    else:
        use_msa_server = False
        msa_server_url = None

    global_settings.update({
        'msa_mode': msa_mode,
        'msa_pairing_strategy': msa_pairing_strategy,
        'use_msa_server': use_msa_server,
        'msa_server_url': msa_server_url,
        'recycling_steps': recycling_steps,
        'sampling_steps': sampling_steps,
        'diffusion_samples': diffusion_samples,
        'max_parallel_samples': max_parallel_samples,
        'step_scale': step_scale,
        'predict_affinity': predict_affinity,
        'affinity_mw_correction': affinity_mw_correction,
        'sampling_steps_affinity': sampling_steps_affinity,
        'diffusion_samples_affinity': diffusion_samples_affinity,
        'output_format': output_format,
        'write_full_pae': write_full_pae,
        'write_full_pde': write_full_pde,
        'use_potentials': use_potentials,
        'seed': seed,
        'write_embeddings': write_embeddings,
        'max_msa_seqs': max_msa_seqs,
        'subsample_msa': subsample_msa,
        'num_subsampled_msa': num_subsampled_msa,
        'skip_existing': skip_existing,
        'enable_ipsae': enable_ipsae,
        'ipsae_png': ipsae_png,
        'ipsae_pdf': ipsae_pdf,
        'ipsae_per_residue': ipsae_per_residue,
        'ipsae_per_contact': ipsae_per_contact,
        'ipsae_pae_cutoff': ipsae_pae_cutoff,
        'ipsae_dist_cutoff': ipsae_dist_cutoff,
        'ipsae_select_residues': ipsae_select_residues,
        'ipsae_batch_analysis': ipsae_batch_analysis,
    })

    print(f"{chr(0x2705)} All settings configured:")
    print(f"  MSA mode: {msa_mode}")
    print(f"  MSA pairing: {msa_pairing_strategy}")
    print(f"  Recycling steps: {recycling_steps}")
    print(f"  Sampling steps: {sampling_steps}")
    print(f"  Diffusion samples: {diffusion_samples}")
    print(f"  Predict affinity: {predict_affinity}")
    print(f"  Output format: {output_format}")
    print(f"  Use potentials: {use_potentials}")
    print(f"  Seed: {seed if seed else '(non-deterministic)'}")
    print(f"  Write embeddings: {write_embeddings}")
    print(f"  Max MSA seqs: {max_msa_seqs}")
    print(f"  Subsample MSA: {subsample_msa}" + (f" ({num_subsampled_msa} seqs)" if subsample_msa else ""))

print(f"\nInterface Analysis (ipSAE):")
print(f"  - Enabled: {enable_ipsae}")
if enable_ipsae:
    print(f"  - PNG: {ipsae_png} | PDF: {ipsae_pdf}")
    print(f"  - Per-residue: {ipsae_per_residue} | Per-contact: {ipsae_per_contact}")
    print(f"  - Cutoffs: PAE={ipsae_pae_cutoff}, dist={ipsae_dist_cutoff}")
    if ipsae_select_residues:
        print(f"  - Selected residues: {ipsae_select_residues}")
    print(f"  - Final batch comparison: {ipsae_batch_analysis}")


In [ ]:
#@title Cell 6: Run Batch Predictions with Calibration-Based Parallel GPU Scheduling
#@markdown **Runs job 1 alone to calibrate VRAM usage, then launches remaining jobs in parallel when GPU memory allows.**
#@markdown Results are automatically uploaded to Google Drive after each job completes.
%%time

import subprocess
import os
import zipfile
import shutil
import time
import gc
import threading
import queue
from datetime import datetime


def get_unique_job_dir(job_name):
    """Return a unique job directory, appending _2, _3, etc. if exists."""
    if not os.path.exists(job_name):
        return job_name
    n = 2
    while os.path.exists(f"{job_name}_{n}"):
        n += 1
    return f"{job_name}_{n}"

def has_existing_output(job_name, search_dir="/content"):
    """Return True if job_name/ contains any .cif or .pdb structure files."""
    job_dir = os.path.join(search_dir, job_name)
    if not os.path.isdir(job_dir):
        return False
    for root, _, files in os.walk(job_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb')):
                return True
    return False
# ============================================================
# VERIFY SETUP
# ============================================================
if 'global_settings' not in globals():
    print(f"{chr(0x274c)} Error: Please run the previous configuration cells first")
    raise ValueError("Configuration not found")

if not global_settings.get('batch_jobs'):
    print(f"{chr(0x274c)} Error: No batch jobs configured. Run CSV upload cell first")
    raise ValueError("No jobs to process")

# ============================================================
# PARALLEL EXECUTION SETTINGS
# ============================================================
enable_parallel = True #@param {type:"boolean"}
#@markdown - **Enable parallel execution**: Launch multiple jobs simultaneously when VRAM allows
#@markdown > **Note**: Jobs are automatically sorted largest-first for optimal calibration. The largest job calibrates VRAM, giving the most accurate per-token rate for parallel scheduling.

vram_limit = 0.85 #@param {type:"slider", min:0.6, max:0.95, step:0.05}
#@markdown - **VRAM limit**: Max fraction of GPU memory to use (0.85 = 85%). Jobs won't launch if this would be exceeded.

# ============================================================
# GPU VERIFICATION
# ============================================================
print("=" * 60)
print(f"{chr(0x1f50d)} GPU VERIFICATION")
print("=" * 60)
gpu_available = False
total_gpu_vram = 0
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        total_gpu_vram = gpu_memory_gb
        print(f"{chr(0x2705)} GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)")
        print(f"   VRAM limit ({vram_limit*100:.0f}%): {gpu_memory_gb * vram_limit:.1f} GB")

        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        torch.cuda.synchronize()
        gpu_available = True

        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            if result.returncode == 0:
                initial_mem = float(result.stdout.strip()) / 1024
                print(f"   Current Usage: {initial_mem:.2f} GB")
                print(f"   {chr(0x2705)} GPU monitoring ready")
            else:
                print(f"   {chr(0x26a0)}  nvidia-smi not available, parallel disabled")
                enable_parallel = False
        except Exception as e:
            print(f"   {chr(0x26a0)}  nvidia-smi failed: {e}")
            enable_parallel = False
    else:
        print(f"{chr(0x26a0)}  WARNING: No GPU detected")
        enable_parallel = False
except Exception as e:
    print(f"{chr(0x26a0)}  GPU test failed: {e}")
    gpu_available = False
    enable_parallel = False

# Kernel test status
if not global_settings.get('kernels_tested', False):
    print(f"\n{chr(0x26a0)}  Kernel preflight test not run - using --no_kernels by default")
    global_settings['use_no_kernels'] = True

# Blackwell: enable mixed kernel mode (triatt=cuEquivariance, trimul=PyTorch)
if global_settings.get('is_blackwell', False):
    os.environ['BOLTZ_DISABLE_TRIMUL_KERNEL'] = '1'
    print(f"{chr(0x2699)}  Blackwell mixed-mode: BOLTZ_DISABLE_TRIMUL_KERNEL=1")

use_no_kernels_flag = global_settings.get('use_no_kernels', True)
print(f"\n{chr(0x1f527)} Kernel mode: {'--no_kernels (CPU fallback)' if use_no_kernels_flag else 'WITH CUDA kernels'}")

# Check ipSAE_batch availability
ipsae_available = False
try:
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True, timeout=10)
    ipsae_available = (result.returncode == 0)
except Exception:
    pass
if global_settings.get('enable_ipsae') and not ipsae_available:
    print("WARNING: ipSAE_batch not installed. Interface analysis will be skipped.")

# ============================================================
# GPU MONITOR
# ============================================================
class GPUMonitor:
    def __init__(self):
        self.monitoring = False
        self.peak_memory = 0
        self.current_memory = 0
        self.monitor_thread = None
        self._lock = threading.Lock()
        self.gpu_available = self._test()
        self._pid_registry = {}  # shell_pid -> {name, peak_gb}

    def _test(self):
        try:
            r = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True, timeout=2)
            return r.returncode == 0
        except:
            return False

    def _get_mem(self):
        if not self.gpu_available:
            return 0
        try:
            r = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True, timeout=2)
            if r.returncode == 0:
                return float(r.stdout.strip()) / 1024
        except:
            pass
        return 0

    def _loop(self):
        while self.monitoring:
            mem = self._get_mem()
            # Per-PID VRAM tracking
            job_mem = {}
            with self._lock:
                need_pid = bool(self._pid_registry)
                shells = set(self._pid_registry.keys()) if need_pid else set()
            if need_pid:
                pid_mem = self._get_pid_mem()
                for gpu_pid, gb in pid_mem.items():
                    shell = self._find_job_for_gpu_pid(gpu_pid, shells)
                    if shell is not None:
                        job_mem[shell] = job_mem.get(shell, 0) + gb
            with self._lock:
                self.current_memory = mem
                if mem > self.peak_memory:
                    self.peak_memory = mem
                for spid, cur_gb in job_mem.items():
                    if spid in self._pid_registry:
                        if cur_gb > self._pid_registry[spid]['peak_gb']:
                            self._pid_registry[spid]['peak_gb'] = cur_gb
            time.sleep(0.5)

    def start(self):
        if not self.gpu_available:
            return
        with self._lock:
            self.monitoring = True
            self.peak_memory = 0
            self.current_memory = 0
        self.monitor_thread = threading.Thread(target=self._loop, daemon=True)
        self.monitor_thread.start()

    def stop(self):
        self.monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=2)
        return self.peak_memory

    def get_current(self):
        with self._lock:
            return self.current_memory

    def get_peak(self):
        with self._lock:
            return self.peak_memory

    def reset_peak(self):
        with self._lock:
            self.peak_memory = self.current_memory

    def _get_pid_mem(self):
        """Get per-PID GPU memory {pid: GB}."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if r.returncode == 0 and r.stdout.strip():
                result = {}
                for ln in r.stdout.strip().split('\n'):
                    parts = ln.strip().split(',')
                    if len(parts) >= 2:
                        try:
                            result[int(parts[0].strip())] = float(parts[1].strip()) / 1024
                        except ValueError:
                            pass
                return result
        except:
            pass
        return {}

    def _find_job_for_gpu_pid(self, gpu_pid, registered_shells):
        """Walk process tree up to find which registered job owns this GPU PID."""
        current = gpu_pid
        visited = set()
        while current > 1 and current not in visited:
            visited.add(current)
            if current in registered_shells:
                return current
            try:
                with open(f'/proc/{current}/status') as f:
                    for line in f:
                        if line.startswith('PPid:'):
                            current = int(line.split(':')[1].strip())
                            break
                    else:
                        break
            except (FileNotFoundError, PermissionError, ValueError):
                break
        return None

    def register_pid(self, shell_pid, job_name):
        """Register a job process PID for per-job VRAM tracking."""
        with self._lock:
            self._pid_registry[shell_pid] = {'name': job_name, 'peak_gb': 0.0}

    def unregister_pid(self, shell_pid):
        """Unregister and return peak VRAM (GB) for a job."""
        with self._lock:
            entry = self._pid_registry.pop(shell_pid, None)
            return entry['peak_gb'] if entry else 0.0

    def get_pid_peak(self, shell_pid):
        """Get peak VRAM (GB) for a specific registered job."""
        with self._lock:
            entry = self._pid_registry.get(shell_pid)
            return entry['peak_gb'] if entry else 0.0


gpu_monitor = GPUMonitor()

def clear_gpu_cache():
    if not gpu_available:
        return
    try:
        import torch
        torch.cuda.empty_cache()
        gc.collect()
    except:
        pass

# ============================================================
# GOOGLE DRIVE HELPERS
# ============================================================
def find_or_create_folder(drive, folder_name, parent_id='root'):
    if not drive:
        return None
    try:
        file_list = drive.ListFile({
            'q': f"title='{folder_name}' and '{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
        }).GetList()
        if file_list:
            print(f"{chr(0x2705)} Found existing folder: {folder_name}")
            return file_list[0]['id']
        folder = drive.CreateFile({
            'title': folder_name, 'mimeType': 'application/vnd.google-apps.folder',
            'parents': [{'id': parent_id}]
        })
        folder.Upload()
        print(f"{chr(0x2705)} Created folder: {folder_name}")
        return folder['id']
    except Exception as e:
        print(f"{chr(0x274c)} Error with folder '{folder_name}': {e}")
        return None

def upload_to_gdrive(drive, file_path, folder_id):
    if not drive or not os.path.exists(file_path):
        return None
    try:
        uploaded = drive.CreateFile({
            'title': os.path.basename(file_path),
            'parents': [{'id': folder_id}]
        })
        uploaded.SetContentFile(file_path)
        uploaded.Upload()
        url = f"https://drive.google.com/file/d/{uploaded['id']}/view"
        print(f"  {chr(0x2601)}  Uploaded: {url}")
        return url
    except Exception as e:
        print(f"  {chr(0x26a0)}  Upload failed: {e}")
        return None

# ============================================================
# NON-BLOCKING OUTPUT READER
# ============================================================

# ============================================================
# MSA FOLDER LOOKUP HELPER
# ============================================================
def find_precomputed_msas(job, msa_folder):
    """Look up pre-computed MSA files for ALL protein chains in a job.

    All-or-nothing: returns MSA paths only if EVERY protein chain has a
    pre-computed MSA file. If any chain is missing, returns empty dict
    and the job falls back to online MSA search entirely.

    Handles naming mismatches between the MSA notebook (raw jobname) and
    folding notebooks (sanitized jobname) by normalizing case and punctuation.

    Args:
        job: Job dict with 'name' and 'sequences'
        msa_folder: Path to paired/ MSA folder

    Returns:
        dict: {chain_name: msa_file_path} for ALL protein chains, or {} if any missing
    """
    if not msa_folder or not os.path.isdir(msa_folder):
        return {}

    job_name = job['name']
    found = {}
    missing = []

    # Collect unique protein chain names
    seen = set()
    protein_chains = []
    for seq in job.get('sequences', []):
        if seq.get('type', '').lower() != 'protein':
            continue
        chain_name = seq.get('name', '')
        if not chain_name or chain_name in seen:
            continue
        seen.add(chain_name)
        protein_chains.append(chain_name)

    if not protein_chains:
        return {}

    # Build index of actual files in the directory.
    # Normalize filenames (lowercase + hyphens to underscores) so that
    # MSA notebook output (raw jobname e.g. "Rx-ADP") matches folding
    # notebook lookups (sanitized e.g. "rx_adp").
    available_files = {}
    for fname in os.listdir(msa_folder):
        if fname.endswith('_paired.a3m'):
            normalized = fname.lower().replace('-', '_')
            available_files[normalized] = fname

    for chain_name in protein_chains:
        expected = f"{job_name}_{chain_name}_paired.a3m"
        exact_path = os.path.join(msa_folder, expected)
        if os.path.isfile(exact_path):
            found[chain_name] = exact_path
        else:
            # Normalized lookup (handles case + hyphen/underscore differences)
            norm_key = expected.lower().replace('-', '_')
            if norm_key in available_files:
                found[chain_name] = os.path.join(msa_folder, available_files[norm_key])
            else:
                missing.append(chain_name)

    if missing:
        print(f"   Pre-computed MSAs missing for: {missing} -- using online MSA for all chains")
        return {}

    print(f"   Pre-computed MSAs found for all {len(found)} chain(s): {list(found.keys())}")
    return found

def _reader_thread(pipe, output_queue):
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line.rstrip('\n'))
    except:
        pass
    finally:
        try:
            pipe.close()
        except:
            pass

# ============================================================
# TOKEN COUNTING
# ============================================================
def count_job_tokens(job, processor):
    total = 0
    for seq in job.get('sequences', []):
        st = seq.get('type', '').lower()
        if st in ['protein', 'dna', 'rna']:
            total += len(seq.get('sequence', ''))
            mods = seq.get('modifications')
            if mods:
                for m in mods:
                    code = m.get('ptmType') or m.get('modificationType', '')
                    if st == 'protein' and hasattr(processor, 'ptm_lookup') and code in processor.ptm_lookup:
                        total += processor.ptm_lookup[code].get('atom_count', 0)
        elif st == 'ligand':
            if seq.get('ccd'):
                total += 20
            elif seq.get('smiles'):
                total += 30
    return max(total, 1)

# ============================================================
# BUILD BOLTZ COMMAND
# ============================================================
def build_boltz_cmd(yaml_file, job_dir, settings):
    cmd_parts = [
        "boltz", "predict", yaml_file,
        "--out_dir", job_dir,
        "--recycling_steps", str(settings.get('recycling_steps', 6)),
        "--sampling_steps", str(settings.get('sampling_steps', 200)),
        "--diffusion_samples", str(settings.get('diffusion_samples', 5)),
        "--max_parallel_samples", str(settings.get('max_parallel_samples', 5)),
        "--step_scale", str(settings.get('step_scale', 1.5)),
        "--output_format", settings.get('output_format', 'mmcif'),
        "--max_msa_seqs", str(settings.get('max_msa_seqs', 8192))
    ]
    if settings.get('use_no_kernels', True):
        cmd_parts.append("--no_kernels")
    if settings.get('use_msa_server', True) and not settings.get('msa_folder'):
        cmd_parts.extend([
            "--use_msa_server",
            "--msa_server_url", settings.get('msa_server_url', 'https://api.colabfold.com'),
            "--msa_pairing_strategy", settings.get('msa_pairing_strategy', 'greedy')
        ])
    if settings.get('write_full_pae', False):
        cmd_parts.append("--write_full_pae")
    if settings.get('write_full_pde', False):
        cmd_parts.append("--write_full_pde")
    if settings.get('use_potentials', False):
        cmd_parts.append("--use_potentials")
    if settings.get('predict_affinity', False):
        cmd_parts.extend([
            "--predict_affinity",
            "--sampling_steps_affinity", str(settings.get('sampling_steps_affinity', 200)),
            "--diffusion_samples_affinity", str(settings.get('diffusion_samples_affinity', 5))
        ])
        if settings.get('affinity_mw_correction', False):
            cmd_parts.append("--affinity_mw_correction")
    seed_val = settings.get('seed', '')
    if seed_val and str(seed_val).strip():
        cmd_parts.extend(["--seed", str(seed_val).strip()])
    if settings.get('write_embeddings', False):
        cmd_parts.append("--write_embeddings")
    if settings.get('subsample_msa', False):
        cmd_parts.extend(["--subsample_msa", "--num_subsampled_msa", str(settings.get('num_subsampled_msa', 1024))])
    return " ".join(cmd_parts)


# ============================================================
# ipSAE INTERFACE ANALYSIS
# ============================================================
def run_ipsae_analysis(job_dir, job_name):
    """Run ipSAE per-job analysis. Returns True on success, False on failure."""
    if not ipsae_available or not settings.get('enable_ipsae'):
        return False

    ipsae_output = os.path.join(job_dir, "ipsae_analysis")
    os.makedirs(ipsae_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", job_dir,
        "--backend", "boltz2",
        "--output_dir", ipsae_output,
        "--pae_cutoff", str(settings.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(settings.get('ipsae_dist_cutoff', 10.0)),
        "--workers", "1",
        "--cores", "1",
    ]
    if settings.get('ipsae_png'):
        cmd_parts.append("--png")
    if settings.get('ipsae_pdf'):
        cmd_parts.append("--pdf")
    if settings.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if settings.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if settings.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", settings['ipsae_select_residues']])

    print(f"   Running ipSAE analysis...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            n_files = sum(1 for f in os.listdir(ipsae_output) if not f.startswith('.'))
            print(f"   ipSAE complete: {n_files} output files")
            return True
        else:
            print(f"   ipSAE failed (rc={result.returncode}): {result.stderr[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print(f"   ipSAE timed out (300s)")
        return False
    except Exception as e:
        print(f"   ipSAE error: {e}")
        return False

# ============================================================
# PROCESS COMPLETED JOB
# ============================================================
def process_boltz_completed(job_name, job_dir, job_time, peak_gpu, drive, folder_id, yaml_file=None):
    results_dirs = [d for d in os.listdir(job_dir) if d.startswith('boltz_results_')]
    if not results_dirs:
        print(f"\n{chr(0x274c)} No results directory for {job_name}")
        print(f"   Contents: {os.listdir(job_dir)}")
        return {'success': False, 'name': job_name, 'error': 'No boltz_results_ dir', 'time': job_time, 'gpu_peak': peak_gpu}

    predictions_dir = os.path.join(job_dir, results_dirs[0])
    structure_files = []
    for root, dirs, files in os.walk(predictions_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb', '.mmcif')):
                structure_files.append(os.path.join(root, f))

    if not structure_files:
        print(f"\n{chr(0x274c)} No structure files for {job_name}")
        for root, dirs, files in os.walk(predictions_dir):
            level = root.replace(predictions_dir, '').count(os.sep)
            indent = '   ' * (level + 1)
            print(f"{indent}{os.path.basename(root)}/")
            for f in files[:5]:
                print(f"{indent}  {f}")
        return {'success': False, 'name': job_name, 'error': 'No structure files', 'time': job_time, 'gpu_peak': peak_gpu}

    print(f"   {chr(0x2705)} {len(structure_files)} structure files")
    for sf in structure_files[:3]:
        print(f"      {chr(0x1f4c4)} {os.path.basename(sf)}")

    # --- ipSAE interface analysis (before zipping) ---
    ipsae_ran = False
    try:
        ipsae_ran = run_ipsae_analysis(job_dir, job_name)
    except Exception as e:
        print(f"   ipSAE error (non-fatal): {e}")

    # ZIP
    zip_filename = f"{job_name}_results.zip"
    zip_path = os.path.join(job_dir, zip_filename)
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(predictions_dir):
                for f in files:
                    fp = os.path.join(root, f)
                    zipf.write(fp, os.path.relpath(fp, predictions_dir))
            # Add input YAML
            if yaml_file and os.path.exists(yaml_file):
                zipf.write(yaml_file, os.path.basename(yaml_file))
                # Add environment log for traceability
                if os.path.isfile("/content/environment.txt"):
                    zipf.write("/content/environment.txt", "environment.txt")
            # Add ipSAE analysis files
            ipsae_dir = os.path.join(job_dir, "ipsae_analysis")
            if os.path.isdir(ipsae_dir):
                for root, dirs, files_list in os.walk(ipsae_dir):
                    for f in files_list:
                        fp = os.path.join(root, f)
                        arcname = os.path.join("ipsae_analysis", os.path.relpath(fp, ipsae_dir))
                        zipf.write(fp, arcname)
        print(f"   {chr(0x1f4e6)} Created: {zip_filename}")
    except Exception as e:
        print(f"   {chr(0x26a0)}  ZIP failed: {e}")

    url = None
    if drive and folder_id:
        url = upload_to_gdrive(drive, zip_path, folder_id)
    else:
        print(f"   {chr(0x1f4c1)} Saved locally: {zip_path}")

    return {
        'success': True, 'name': job_name, 'structures': len(structure_files),
        'time': job_time, 'url': url, 'gpu_peak': peak_gpu,
        'ipsae_ran': ipsae_ran,
    }

# ============================================================
# SEQUENTIAL RUNNER
# ============================================================
def run_boltz_sequential(job, job_idx, total_jobs, settings, drive, folder_id):
    job_start = time.time()
    print(f"\n{'='*60}")
    print(f"{chr(0x1f680)} Job {job_idx}/{total_jobs}: {job['name']}")
    print(f"{'='*60}")

    gpu_monitor.start()
    time.sleep(0.5)

    job_name = job['name']
    if global_settings.get('skip_existing', False) and has_existing_output(job_name):
        print(f"  \u23ed Skipping {job_name}: output already exists")
        gpu_monitor.stop()
        return {'name': job_name, 'skipped': True, 'success': True,
                'structures': 0, 'time': 0, 'gpu_peak': 0,
                'tokens': count_job_tokens(job, settings['processor'])}
    job_dir = get_unique_job_dir(job_name)
    os.makedirs(job_dir, exist_ok=True)

    yaml_content = settings['processor']._generate_yaml(job)

    # Inject pre-computed MSA paths if available
    msa_folder_val = global_settings.get('msa_folder', '')
    if msa_folder_val:
        precomp_msas = find_precomputed_msas(job, msa_folder_val)
        if precomp_msas:
            import yaml as _yaml
            yaml_data = _yaml.safe_load(yaml_content)
            # Build chain_id -> chain_name mapping from job sequences
            id_to_name = {}
            for seq in job.get('sequences', []):
                if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                    id_to_name[seq.get('id', '')] = seq['name']
            for entry in yaml_data.get('sequences', []):
                if 'protein' in entry:
                    protein = entry['protein']
                    chain_id = protein.get('id')
                    ids = chain_id if isinstance(chain_id, list) else [chain_id]
                    for cid in ids:
                        chain_name = id_to_name.get(cid)
                        if chain_name and chain_name in precomp_msas:
                            protein['msa'] = precomp_msas[chain_name]
                            break
            yaml_content = _yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
    yaml_file = os.path.join(job_dir, f"{job_name}.yaml")
    with open(yaml_file, 'w') as f:
        f.write(yaml_content)

    cmd = build_boltz_cmd(yaml_file, job_dir, settings)
    print(f"{chr(0x1f527)} Command: {cmd}")
    print(f"\n{chr(0x2699)}  Running Boltz-2 prediction...")

    try:
        proc = subprocess.Popen(
            cmd, shell=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )

        out_q = queue.Queue()
        reader = threading.Thread(target=_reader_thread, args=(proc.stdout, out_q), daemon=True)
        reader.start()

        while proc.poll() is None:
            while not out_q.empty():
                try:
                    line = out_q.get_nowait()
                    if line:
                        ll = line.lower()
                        if any(kw in ll for kw in ['progress', 'step', 'recycling', 'diffusion',
                                                     'completed', 'saving', 'loading', 'predicting']):
                            cur = gpu_monitor.get_current()
                            if cur > 1.0:
                                print(f"   {line[:100]} [GPU: {cur:.1f} GB]")
                            else:
                                print(f"   {line[:100]}")
                        elif 'error' in ll or 'warning' in ll:
                            print(f"   {chr(0x26a0)}  {line[:100]}")
                except queue.Empty:
                    break
            time.sleep(0.5)

        # Drain
        while not out_q.empty():
            try:
                line = out_q.get_nowait()
                if line and ('error' in line.lower() or 'completed' in line.lower()):
                    print(f"   {line[:100]}")
            except queue.Empty:
                break

        rc = proc.wait(timeout=60)
        peak_gpu = gpu_monitor.stop()

        print(f"\n{chr(0x1f3ae)} GPU Peak: {peak_gpu:.2f} GB")

        if rc != 0:
            print(f"\n{chr(0x274c)} Failed (exit code {rc})")
            return {
                'success': False, 'name': job_name, 'error': f'Exit code {rc}',
                'time': time.time() - job_start, 'gpu_peak': peak_gpu
            }

    except subprocess.TimeoutExpired:
        proc.kill()
        peak_gpu = gpu_monitor.stop()
        return {'success': False, 'name': job_name, 'error': 'Timeout', 'time': time.time() - job_start, 'gpu_peak': peak_gpu}
    except Exception as e:
        peak_gpu = gpu_monitor.stop()
        return {'success': False, 'name': job_name, 'error': str(e), 'time': time.time() - job_start, 'gpu_peak': peak_gpu}

    job_time = time.time() - job_start
    result = process_boltz_completed(job_name, job_dir, job_time, peak_gpu, drive, folder_id, yaml_file=yaml_file)
    if 'tokens' not in result:
        result['tokens'] = count_job_tokens(job, settings['processor'])
    print(f"\n{chr(0x23f1)}  {job_time:.1f}s ({job_time/60:.1f} min)")
    return result

# ============================================================
# SETUP
# ============================================================
settings = global_settings
batch_jobs = settings['batch_jobs']
processor = settings['processor']
# Sort jobs largest-first for optimal calibration accuracy
batch_jobs = sorted(batch_jobs, key=lambda j: count_job_tokens(j, processor), reverse=True)
print(f"Jobs sorted by size (largest first):")
for _j in batch_jobs:
    print(f"  {_j['name']}: {count_job_tokens(_j, processor)} tokens")

gdrive_folder_id = None
uploaded_files = []
if settings.get('drive'):
    gdrive_folder_id = find_or_create_folder(settings['drive'], settings.get('gdrive_folder_name', 'Boltz2_Results'))

drive_ref = settings.get('drive')

print(f"\n{'='*60}")
print(f"{chr(0x1f680)} STARTING BATCH PROCESSING: {len(batch_jobs)} JOBS")
print(f"{'='*60}")
print(f"  - Recycling steps: {settings.get('recycling_steps', 6)}")
print(f"  - Sampling steps: {settings.get('sampling_steps', 200)}")
print(f"  - Diffusion samples: {settings.get('diffusion_samples', 5)}")
print(f"  - Kernels: {'disabled (--no_kernels)' if settings.get('use_no_kernels', True) else 'enabled'}")
print(f"  - Parallel mode: {enable_parallel}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if len(batch_jobs) == 1:
    enable_parallel = False

# ============================================================
# MAIN EXECUTION
# ============================================================
batch_start = time.time()
completed_jobs = []
failed_jobs = []

if not enable_parallel or not gpu_available:
    # SEQUENTIAL MODE
    print(f"\n{chr(0x1f504)} SEQUENTIAL MODE")
    for i, job in enumerate(batch_jobs, 1):
        result = run_boltz_sequential(job, i, len(batch_jobs), settings, drive_ref, gdrive_folder_id)
        if result['success']:
            completed_jobs.append(result)
        else:
            failed_jobs.append(result)
        clear_gpu_cache()
        time.sleep(1)
else:
    # CALIBRATION-BASED PARALLEL MODE
    print(f"\n{'='*60}")
    print(f"{chr(0x1f9ea)} CALIBRATION-BASED PARALLEL MODE")
    print(f"{'='*60}")
    print(f"Phase 1: Run job 1 alone to measure VRAM")
    print(f"Phase 2: Estimate remaining, launch in parallel")

    # --- PHASE 1: CALIBRATION (try jobs one by one until one succeeds) ---
    print(f"\n{chr(0x1f4cf)} PHASE 1: CALIBRATION")
    print(f"Running jobs one at a time until one succeeds to measure VRAM...")

    cal_result = None
    cal_peak_vram = 0
    cal_tokens = 0
    calibration_idx = -1
    skipped_in_cal = 0

    for cal_idx, cal_job in enumerate(batch_jobs):
        cal_job_name = cal_job['name']

        # Skip already-completed jobs -- they give no VRAM data
        if global_settings.get('skip_existing', False) and has_existing_output(cal_job_name):
            print(f"  \u23ed Skipping {cal_job_name} (already done)")
            completed_jobs.append({'name': cal_job_name, 'skipped': True, 'success': True,
                                   'structures': 0, 'time': 0, 'gpu_peak': 0,
                                   'tokens': count_job_tokens(cal_job, processor)})
            skipped_in_cal += 1
            continue

        cal_tokens = count_job_tokens(cal_job, processor)
        print(f"\n   Calibration attempt {cal_idx + 1}/{len(batch_jobs)}: {cal_job_name} ({cal_tokens} tokens)")

        cal_result = run_boltz_sequential(cal_job, cal_idx + 1, len(batch_jobs), settings, drive_ref, gdrive_folder_id)

        if cal_result['success']:
            completed_jobs.append(cal_result)
            calibration_idx = cal_idx
            cal_peak_vram = cal_result.get('gpu_peak', 0)
            print(f"\n   {chr(0x2705)} Calibration succeeded on job '{cal_job_name}' -- peak VRAM: {cal_peak_vram:.2f} GB")
            break
        else:
            failed_jobs.append(cal_result)
            print(f"   {chr(0x26a0)}  Job '{cal_job_name}' failed -- trying next job for calibration...")
            clear_gpu_cache()
            time.sleep(2)
            continue

    remaining = batch_jobs[calibration_idx + 1:] if calibration_idx >= 0 else []

    if calibration_idx < 0:
        if skipped_in_cal == len(batch_jobs):
            print(f"\n{chr(0x2705)} All {len(batch_jobs)} jobs already completed. Nothing to run.")
        else:
            print(f"\n{chr(0x274c)} ALL {len(batch_jobs)} jobs failed. No successful calibration possible.")
    elif not remaining:
        print(f"\n{chr(0x2705)} Only one job - done.")
    elif cal_peak_vram <= 0:
        print(f"\n{chr(0x26a0)}  No VRAM data. Sequential fallback.")
        for i, job in enumerate(remaining, calibration_idx + 2):
            r = run_boltz_sequential(job, i, len(batch_jobs), settings, drive_ref, gdrive_folder_id)
            (completed_jobs if r['success'] else failed_jobs).append(r)
            clear_gpu_cache(); time.sleep(1)
    elif cal_peak_vram > total_gpu_vram * vram_limit:
        print(f"\n{chr(0x26a0)}  Calibration used {cal_peak_vram:.1f} GB (>{total_gpu_vram * vram_limit:.1f} GB limit)")
        print(f"   Sequential for remaining {len(remaining)} jobs.")
        for i, job in enumerate(remaining, calibration_idx + 2):
            r = run_boltz_sequential(job, i, len(batch_jobs), settings, drive_ref, gdrive_folder_id)
            (completed_jobs if r['success'] else failed_jobs).append(r)
            clear_gpu_cache(); time.sleep(1)
    else:
        # Phase 2: Estimate
        vram_per_token = cal_peak_vram / cal_tokens
        model_overhead = cal_peak_vram * 0.3

        print(f"\n{chr(0x1f4ca)} PHASE 2: ESTIMATION")
        print(f"   Rate: {vram_per_token:.4f} GB/token, floor: {model_overhead:.2f} GB")

        job_estimates = []
        for job in remaining:
            tokens = count_job_tokens(job, processor)
            est = max(vram_per_token * tokens * 1.15, model_overhead)
            job_estimates.append({'job': job, 'tokens': tokens, 'estimated_vram': est})
            print(f"   {job['name']}: {tokens} tokens -> ~{est:.1f} GB")

        # Phase 3: Parallel
        print(f"\n{chr(0x1f680)} PHASE 3: PARALLEL EXECUTION")
        safe_limit = total_gpu_vram * vram_limit
        emergency = total_gpu_vram * vram_limit

        pending = list(enumerate(job_estimates, calibration_idx + 2))
        running = []
        emergency_triggered = False

        gpu_monitor.start()

        def try_launch():
            if not pending:
                return False
            cur = gpu_monitor.get_current()
            # Use HIGHER of actual GPU or estimated running (nvidia-smi lags behind)
            estimated_running = sum(p['est_vram'] for p in running)
            effective_vram = max(cur, estimated_running)
            jn, est = pending[0]
            if effective_vram + est['estimated_vram'] < safe_limit:
                pending.pop(0)
                job = est['job']
                jname = job['name']
                if global_settings.get('skip_existing', False) and has_existing_output(jname):
                    print(f"  \u23ed Skipping {jname}: output already exists")
                    completed_jobs.append({'name': jname, 'skipped': True, 'success': True,
                                    'structures': 0, 'time': 0, 'gpu_peak': 0,
                                    'tokens': est['tokens']})
                    return True
                jdir = get_unique_job_dir(jname)
                os.makedirs(jdir, exist_ok=True)
                yaml_content = processor._generate_yaml(job)

                # Inject pre-computed MSA paths if available
                msa_folder_val = global_settings.get('msa_folder', '')
                if msa_folder_val:
                    precomp_msas = find_precomputed_msas(job, msa_folder_val)
                    if precomp_msas:
                        import yaml as _yaml
                        yaml_data = _yaml.safe_load(yaml_content)
                        id_to_name = {}
                        for seq in job.get('sequences', []):
                            if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                                id_to_name[seq.get('id', '')] = seq['name']
                        for entry in yaml_data.get('sequences', []):
                            if 'protein' in entry:
                                protein = entry['protein']
                                chain_id = protein.get('id')
                                ids = chain_id if isinstance(chain_id, list) else [chain_id]
                                for cid in ids:
                                    chain_name = id_to_name.get(cid)
                                    if chain_name and chain_name in precomp_msas:
                                        protein['msa'] = precomp_msas[chain_name]
                                        break
                        yaml_content = _yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
                yf = os.path.join(jdir, f"{jname}.yaml")
                with open(yf, 'w') as f:
                    f.write(yaml_content)
                cmd = build_boltz_cmd(yf, jdir, settings)
                try:
                    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                        text=True, bufsize=1, universal_newlines=True)
                except Exception as e:
                    print(f"{chr(0x274c)} Launch failed for {jname}: {e}")
                    failed_jobs.append({'success': False, 'name': jname, 'error': str(e), 'time': 0, 'gpu_peak': 0})
                    return try_launch()
                oq = queue.Queue()
                t = threading.Thread(target=_reader_thread, args=(p.stdout, oq), daemon=True)
                t.start()
                running.append({
                    'process': p, 'job': job, 'name': jname, 'dir': jdir, 'yaml': yf,
                    'number': jn, 'tokens': est['tokens'], 'est_vram': est['estimated_vram'],
                    'start': time.time(), 'queue': oq
                })
                gpu_monitor.register_pid(p.pid, jname)
                print(f"\n{chr(0x1f680)} Launched {jn}/{len(batch_jobs)}: {jname} (est. {est['estimated_vram']:.1f} GB, GPU: {effective_vram:.1f}/{safe_limit:.1f} GB)")
                time.sleep(2)
                return True
            return False

        while try_launch():
            pass

        if running:
            print(f"\n{chr(0x2705)} {len(running)} running, {len(pending)} pending")

        while running or pending:
            for info in list(running):
                while not info['queue'].empty():
                    try:
                        line = info['queue'].get_nowait()
                        if line:
                            ll = line.lower()
                            if any(kw in ll for kw in ['progress', 'step', 'completed', 'saving']):
                                print(f"   [{info['name']}] {line[:80]} [GPU: {gpu_monitor.get_current():.1f} GB]")
                            elif 'error' in ll or 'warning' in ll:
                                print(f"   {chr(0x26a0)} [{info['name']}] {line[:80]}")
                    except queue.Empty:
                        break

                rc = info['process'].poll()
                if rc is not None:
                    running.remove(info)
                    jt = time.time() - info['start']
                    pid_peak = gpu_monitor.unregister_pid(info['process'].pid)

                    # Final drain: capture any lines buffered between last poll and exit
                    while not info['queue'].empty():
                        try:
                            info['queue'].get_nowait()
                        except queue.Empty:
                            break

                    try:
                        if rc == 0:
                            print(f"\n{chr(0x2705)} Completed: {info['name']} ({jt:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            r = process_boltz_completed(info['name'], info['dir'], jt, pid_peak, drive_ref, gdrive_folder_id, yaml_file=info.get('yaml'))
                            r['tokens'] = info['tokens']
                            (completed_jobs if r['success'] else failed_jobs).append(r)
                        else:
                            print(f"\n{chr(0x274c)} Failed: {info['name']} (exit {rc}, {jt:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            failed_jobs.append({'success': False, 'name': info['name'], 'error': f'Exit {rc}', 'time': jt, 'gpu_peak': pid_peak})
                    except Exception as e:
                        print(f"  ERROR processing {info['name']}: {e}")
                        failed_jobs.append({'success': False, 'name': info['name'], 'error': str(e), 'time': jt, 'gpu_peak': pid_peak})

                    clear_gpu_cache()
                    time.sleep(1)
                    gpu_monitor.reset_peak()
                    if not emergency_triggered:
                        while try_launch():
                            pass
                        if running or pending:
                            print(f"   {chr(0x1f4ca)} {len(running)} running, {len(pending)} pending, {len(completed_jobs)} done")

            cur = gpu_monitor.get_current()
            if cur > emergency and not emergency_triggered:
                emergency_triggered = True
                print(f"\n{chr(0x1f6a8)} EMERGENCY: VRAM {cur:.1f}/{total_gpu_vram:.1f} GB - stopping launches")

            time.sleep(2)

        gpu_monitor.stop()

        if pending:
            print(f"\n{chr(0x1f504)} Running {len(pending)} remaining sequentially...")
            for jn, est in pending:
                r = run_boltz_sequential(est['job'], jn, len(batch_jobs), settings, drive_ref, gdrive_folder_id)
                (completed_jobs if r['success'] else failed_jobs).append(r)
                clear_gpu_cache(); time.sleep(1)


# ============================================================
# FINAL ipSAE BATCH ANALYSIS
# ============================================================
ipsae_batch_url = None
if (ipsae_available and settings.get('enable_ipsae')
        and settings.get('ipsae_batch_analysis') and len(completed_jobs) > 1):
    print("\n" + "=" * 60)
    print("FINAL ipSAE BATCH ANALYSIS")
    print("=" * 60)

    ipsae_batch_output = "/content/ipsae_batch_results"
    os.makedirs(ipsae_batch_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", "/content",
        "--backend", "boltz2",
        "--output_dir", ipsae_batch_output,
        "--pae_cutoff", str(settings.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(settings.get('ipsae_dist_cutoff', 10.0)),
        "--cores", str(os.cpu_count() or 2),
    ]
    if settings.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if settings.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if settings.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", settings['ipsae_select_residues']])

    print(f"Analyzing {len(completed_jobs)} completed jobs...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=600)
        if result.returncode == 0:
            batch_files = [f for f in os.listdir(ipsae_batch_output) if not f.startswith('.')]
            print(f"Batch analysis complete: {len(batch_files)} files")
            for bf in sorted(batch_files):
                size = os.path.getsize(os.path.join(ipsae_batch_output, bf))
                print(f"   {bf} ({size//1024}KB)")

            # Upload to GDrive
            if drive_ref and gdrive_folder_id:
                batch_zip = "/content/ipsae_batch_results.zip"
                with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for root, dirs, files_list in os.walk(ipsae_batch_output):
                        for f in files_list:
                            fp = os.path.join(root, f)
                            arcname = os.path.join("ipsae_batch_results", os.path.relpath(fp, ipsae_batch_output))
                            zipf.write(fp, arcname)
                ipsae_batch_url = upload_to_gdrive(drive_ref, batch_zip, gdrive_folder_id)
                print(f"Uploaded: {ipsae_batch_url}")
            else:
                print(f"Saved locally: {ipsae_batch_output}/")
        else:
            print(f"Batch analysis failed (rc={result.returncode})")
            if result.stderr:
                print(f"   {result.stderr[:300]}")
    except subprocess.TimeoutExpired:
        print("Batch analysis timed out (600s)")
    except Exception as e:
        print(f"Batch analysis error: {e}")

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - batch_start

print(f"\n{'='*60}")
print(f"{chr(0x1f389)} BATCH PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"{chr(0x23f1)}  Duration: {total_time/60:.1f} min ({total_time/3600:.2f} hr)")
print(f"{chr(0x2705)} Successful: {len(completed_jobs)}/{len(batch_jobs)}")

if failed_jobs:
    print(f"{chr(0x274c)} Failed: {len(failed_jobs)}/{len(batch_jobs)}")

if completed_jobs:
    print(f"\n{chr(0x2705)} Successful jobs:")
    for j in completed_jobs:
        t = f"{j['time']:.1f}s" if j['time'] < 60 else f"{j['time']/60:.1f}m"
        g = f" | Peak: {j['gpu_peak']:.2f} GB" if j.get('gpu_peak') else ""
        print(f"  - {j['name']}: {j.get('structures', '?')} structures ({t}{g})")
        if j.get('url'):
            print(f"    {chr(0x1f517)} {j['url']}")

if failed_jobs:
    print(f"\n{chr(0x274c)} Failed jobs:")
    for j in failed_jobs:
        print(f"  - {j['name']}: {j.get('error', 'Unknown')[:80]}")

if gpu_available and completed_jobs:
    peaks = [j['gpu_peak'] for j in completed_jobs if j.get('gpu_peak', 0) > 0]
    if peaks:
        print(f"\n{chr(0x1f3ae)} GPU Stats: avg={sum(peaks)/len(peaks):.2f} GB, max={max(peaks):.2f} GB, capacity={total_gpu_vram:.1f} GB")


# ipSAE Analysis Summary
if settings.get('enable_ipsae'):
    ipsae_count = sum(1 for j in completed_jobs if j.get('ipsae_ran'))
    print(f"\nipSAE Interface Analysis:")
    print(f"   Per-job: {ipsae_count}/{len(completed_jobs)} jobs analyzed")
    if ipsae_batch_url:
        print(f"   Batch comparison: {ipsae_batch_url}")

if completed_jobs:
    avg = sum(j['time'] for j in completed_jobs) / len(completed_jobs)
    print(f"{chr(0x23f1)}  Average: {avg:.1f}s/job ({avg/60:.1f} min)")

if drive_ref and gdrive_folder_id:
    print(f"\n{chr(0x2601)}  Results in: {settings.get('gdrive_folder_name', 'Boltz2_Results')}")
    print(f"   {chr(0x1f4c1)} https://drive.google.com/drive/folders/{gdrive_folder_id}")

print(f"\n{chr(0x1f389)} All done!")
